## 1. Load the data

The first step is to load the Excel workbook and the datasets that will be used throughout the analysis.

The workbook contains several sheets, but the main datasets for Luma are:

- **Kerkesat** — citizen requests and their workflow
- **Operacione_Mbetje** — operational data for public services / waste
- **Operacione_Rruge** — road and infrastructure operations
- **Buxhet_Finance** — budget and spending data

We also load the reference tables:

- **Njesite**
- **Drejtorite**
- **Kategorite**
- **Perdoruesit**

These reference tables will later help us validate whether requests are assigned to valid units, categories, directorates, and users.

In [ ]:
import pandas as pd
import numpy as np

file_path = "../data/raw/Bashkia_360_Data.xlsx"

njesite = pd.read_excel(file_path, sheet_name="Njesite")
drejtorite = pd.read_excel(file_path, sheet_name="Drejtorite")
kategorite = pd.read_excel(file_path, sheet_name="Kategorite")
perdoruesit = pd.read_excel(file_path, sheet_name="Perdoruesit")

kerkesat_raw = pd.read_excel(file_path, sheet_name="Kerkesat")
mbetje_raw = pd.read_excel(file_path, sheet_name="Operacione_Mbetje")
rruge_raw = pd.read_excel(file_path, sheet_name="Operacione_Rruge")
buxhet_raw = pd.read_excel(file_path, sheet_name="Buxhet_Finance")

print("Data loaded successfully")

Data loaded successfully


## 2. Understand the workbook

Before cleaning the data, we need to understand the role of each sheet in the Excel file.

The workbook contains two types of tables:

### Reference tables
These describe the structure of the municipal system:

- **Njesite** — administrative units included in the demo
- **Drejtorite** — municipal directorates
- **Kategorite** — request categories, their responsible directorate, and SLA
- **Perdoruesit** — users, roles, and access responsibilities

### Main operational datasets
These contain the data that will later be cleaned and analyzed:

- **Kerkesat** — citizen requests and their workflow
- **Operacione_Mbetje** — waste/public-service operational data
- **Operacione_Rruge** — road and infrastructure operational data
- **Buxhet_Finance** — budget and spending information

The reference tables will help us validate the main datasets.

In [ ]:
# Group the sheets by their role

reference_tables = {
    "Njesite": njesite,
    "Drejtorite": drejtorite,
    "Kategorite": kategorite,
    "Perdoruesit": perdoruesit,
}

main_datasets = {
    "Kerkesat": kerkesat_raw,
    "Operacione_Mbetje": mbetje_raw,
    "Operacione_Rruge": rruge_raw,
    "Buxhet_Finance": buxhet_raw,
}

print("REFERENCE TABLES")
print("-" * 40)

for name, df in reference_tables.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")

print("\nMAIN DATASETS")
print("-" * 40)

for name, df in main_datasets.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")

REFERENCE TABLES
----------------------------------------
Njesite: 3 rows x 3 columns
Drejtorite: 2 rows x 2 columns
Kategorite: 6 rows x 4 columns
Perdoruesit: 7 rows x 5 columns

MAIN DATASETS
----------------------------------------
Kerkesat: 556 rows x 21 columns
Operacione_Mbetje: 80 rows x 10 columns
Operacione_Rruge: 20 rows x 10 columns
Buxhet_Finance: 8 rows x 7 columns


In [ ]:
for name, df in reference_tables.items():
    print(f"\n{name}")
    display(df.head())

for name, df in main_datasets.items():
    print(f"\n{name}")
    display(df.head())


Njesite


,Kodi,Njësia administrative,Lloji
0,NJ01,Elbasan (qyteti),Qytet
1,NJ02,Bradashesh,Njësi Administrative
2,NJ09,Papër,Njësi Administrative



Drejtorite


,Kodi,Drejtoria
0,DSHP,Drejtoria e Shërbimeve Publike
1,DI,Drejtoria e Infrastrukturës



Kategorite


,Kodi,Kategoria,Drejtoria,Afati (ditë)
0,MBETJE,Mbetje të pagrumbulluara,DSHP,2
1,PASTRIM,Pastrim rrugësh,DSHP,3
2,GJELBERIM,Pemë / gjelbërim,DSHP,10
3,GROPA,Gropa / rrugë e dëmtuar,DI,14
4,NDRICIM,Ndriçim publik,DI,5


... (6 outputs left out)


Operacione_Rruge


,Muaji,Njësia (kodi),Njësia administrative,Lagjja / Fshati,Drejtoria,Incidente të raportuara,Incidente të hapura,Ditë mesatare riparimi,Raste kullimi,Ndërhyrje urgjente
0,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DI,12,1,4.0,1,3
1,2026-07-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DI,13,4,4.6,4,2
2,2026-08-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DI,7,4,4.0,1,2
3,2026-09-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DI,14,3,3.2,4,0
4,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 3,DI,14,2,4.1,2,1



Buxhet_Finance


,Muaji,Kodi drejtoria,Drejtoria,Buxheti planifikuar ALL,Shpenzimi real ALL,Shpenzimi angazhuar ALL,Realizimi %
0,2026-06-01,DSHP,Drejtoria e Shërbimeve Publike,41000000,32900000,35780879,NaN
1,2026-06-01,DI,Drejtoria e Infrastrukturës,50000000,35900000,37181840,NaN
2,2026-07-01,DSHP,Drejtoria e Shërbimeve Publike,42000000,34100000,37141904,NaN
3,2026-07-01,DI,Drejtoria e Infrastrukturës,51000000,37100000,40486799,NaN
4,2026-08-01,DSHP,Drejtoria e Shërbimeve Publike,45000000,41600000,45745565,NaN


### Interpretation

The workbook has a clear relational structure.

The reference tables define the valid municipal structure, while the operational datasets contain the activity that we want to analyze.

For example:

- `Kategorite` tells us which directorate should handle each type of request.
- `Perdoruesit` tells us which users belong to each directorate.
- `Njesite` defines the valid administrative areas.
- `Kerkesat` contains the actual citizen-request workflow.

This means that later we can validate the operational data against the reference tables instead of checking each field manually.

The four main datasets also represent different views of municipal performance:

**Citizen demand → Operations → Infrastructure → Finance**

These will later be connected to build the decision-support layer of Luma.

## 3. Profile each dataset

Before cleaning the data, we need to understand the structure and basic quality of each dataset.

For every main dataset, we will inspect:

- number of rows and columns
- column names
- data types
- missing values
- a small sample of records

This gives us a quick overview of the data and helps us identify which fields need validation or cleaning.

In [ ]:
def profile_dataset(name, df):
    print("=" * 70)
    print(name)
    print("=" * 70)

    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))

    print("\nMissing values:")
    missing = df.isna().sum().to_frame("missing_count")

    missing["missing_pct"] = (
        missing["missing_count"] / len(df) * 100
    ).round(2)

    display(missing)

    print("\nSample records:")
    display(df.head())

    print("\n")


profile_dataset("Kerkesat", kerkesat_raw)
profile_dataset("Operacione_Mbetje", mbetje_raw)
profile_dataset("Operacione_Rruge", rruge_raw)
profile_dataset("Buxhet_Finance", buxhet_raw)

Kerkesat
Rows: 556
Columns: 21

Column names:
['ID', 'Data e regjistrimit', 'Kanali', 'Njësia (kodi)', 'Njësia administrative', 'Lagjja / Fshati', 'Vendi', 'Kategoria', 'Përshkrimi', 'Regjistruar nga', 'Drejtoria', 'Përgjegjësi', 'Data e caktimit', 'Afati (ditë)', 'Data limit', 'Statusi', 'Data e zgjidhjes', 'Shënim i përgjegjësit', 'Ditë trajtimi', 'Brenda afatit?', 'Recurring group']

Data types:


,dtype
ID,object
Data e regjistrimit,datetime64[ns]
Kanali,object
Njësia (kodi),object
Njësia administrative,object
Lagjja / Fshati,object
Vendi,object
Kategoria,object
Përshkrimi,object
Regjistruar nga,object



Missing values:


,missing_count,missing_pct
ID,0,0.00
Data e regjistrimit,0,0.00
Kanali,0,0.00
Njësia (kodi),0,0.00
Njësia administrative,0,0.00
Lagjja / Fshati,0,0.00
Vendi,0,0.00
Kategoria,0,0.00
Përshkrimi,0,0.00
Regjistruar nga,0,0.00



Sample records:


,ID,Data e regjistrimit,Kanali,Njësia (kodi),Njësia administrative,Lagjja / Fshati,Vendi,Kategoria,Përshkrimi,Regjistruar nga,...,Përgjegjësi,Data e caktimit,Afati (ditë),Data limit,Statusi,Data e zgjidhjes,Shënim i përgjegjësit,Ditë trajtimi,Brenda afatit?,Recurring group
0,K-2026-0526,2026-06-01,Telefon,NJ09,Papër,Papër,Stacioni,Ndriçim publik,Ndriçimi mungon pranë shkollës.,operator.telefon,...,pergj.di,2026-06-02,5.0,NaN,E zgjidhur,2026-06-05,NaN,NaN,NaN,NaN
1,K-2026-0221,2026-06-02,Email,NJ01,Elbasan (qyteti),Lagjja nr. 3,Parku qendror,Gropa / rrugë e dëmtuar,Trotuari është dëmtuar dhe është i rrezikshëm ...,operator.sporteli,...,pergj.di,2026-06-03,14.0,NaN,E zgjidhur,2026-06-04,Ndërhyrja u krye dhe rasti u mbyll.,NaN,NaN,NaN
2,K-2026-0229,2026-06-02,Telefon,NJ01,Elbasan (qyteti),Lagjja nr. 3,Pallati nr. 12,Gropa / rrugë e dëmtuar,Gropë e madhe po pengon qarkullimin.,operator.telefon,...,pergj.di,2026-06-02,14.0,NaN,E zgjidhur,2026-06-15,Ndërhyrja u krye dhe rasti u mbyll.,NaN,NaN,NaN
3,K-2026-0063,2026-06-03,Telefon,NJ01,Elbasan (qyteti),Lagjja nr. 2,Tregu i lagjes,Ndriçim publik,Ndriçimi mungon pranë shkollës.,operator.telefon,...,pergj.di,2026-06-03,5.0,NaN,E zgjidhur,2026-06-08,Ndërhyrja u krye dhe rasti u mbyll.,NaN,NaN,NaN
4,K-2026-0107,2026-06-03,Telefon,NJ01,Elbasan (qyteti),Lagjja nr. 3,Pallati nr. 12,Pemë / gjelbërim,Degë të rënda rrezikojnë kalimtarët.,operator.telefon,...,pergj.dshp,2026-06-04,10.0,NaN,E zgjidhur,2026-06-11,NaN,NaN,NaN,NaN


... (14 outputs left out)


Missing values:


,missing_count,missing_pct
Muaji,0,0.0
Kodi drejtoria,0,0.0
Drejtoria,0,0.0
Buxheti planifikuar ALL,0,0.0
Shpenzimi real ALL,0,0.0
Shpenzimi angazhuar ALL,0,0.0
Realizimi %,8,100.0



Sample records:


,Muaji,Kodi drejtoria,Drejtoria,Buxheti planifikuar ALL,Shpenzimi real ALL,Shpenzimi angazhuar ALL,Realizimi %
0,2026-06-01,DSHP,Drejtoria e Shërbimeve Publike,41000000,32900000,35780879,NaN
1,2026-06-01,DI,Drejtoria e Infrastrukturës,50000000,35900000,37181840,NaN
2,2026-07-01,DSHP,Drejtoria e Shërbimeve Publike,42000000,34100000,37141904,NaN
3,2026-07-01,DI,Drejtoria e Infrastrukturës,51000000,37100000,40486799,NaN
4,2026-08-01,DSHP,Drejtoria e Shërbimeve Publike,45000000,41600000,45745565,NaN


### Interpretation

The request dataset contains the expected operational and workflow fields.

The main date columns are already recognized correctly as datetime values.

However, three derived fields require additional validation and cleaning:

- `Data limit` is currently stored as a numeric value instead of a date;
- `Ditë trajtimi` must be verified as the number of days required to handle a request;
- `Brenda afatit?` must be recalculated from the actual deadline and resolution dates.

These fields should not be used directly as source-of-truth values until they are validated and recalculated during preprocessing.

## 4. Check missing values

Missing values are not always data-quality problems.

For example:

- a request that is still open should not have a resolution date;
- a newly registered request may not have an assigned directorate or responsible user yet;
- a resolution note may legitimately be empty.

For this reason, we first measure the missing values and then interpret them according to the business workflow.

In [ ]:
def check_missing_values(name, df):
    print("=" * 70)
    print(name)
    print("=" * 70)

    missing = df.isna().sum().to_frame("missing_count")

    missing["missing_pct"] = (
        missing["missing_count"] / len(df) * 100
    ).round(2)

    missing = missing[
        missing["missing_count"] > 0
    ].sort_values(
        "missing_count",
        ascending=False
    )

    display(missing)

    print()


check_missing_values("Kerkesat", kerkesat_raw)
check_missing_values("Operacione_Mbetje", mbetje_raw)
check_missing_values("Operacione_Rruge", rruge_raw)
check_missing_values("Buxhet_Finance", buxhet_raw)

Kerkesat


,missing_count,missing_pct
Ditë trajtimi,556,100.00
Data limit,556,100.00
Brenda afatit?,556,100.00
Recurring group,532,95.68
Shënim i përgjegjësit,121,21.76
Data e zgjidhjes,74,13.31
Përgjegjësi,28,5.04
Drejtoria,28,5.04
Data e caktimit,28,5.04
Afati (ditë),28,5.04



Operacione_Mbetje


,missing_count,missing_pct



Operacione_Rruge


,missing_count,missing_pct



Buxhet_Finance


,missing_count,missing_pct
Realizimi %,8,100.0


### Interpretation

The missing-value analysis shows two different situations.

#### 1. Derived fields are completely empty

The following fields are 100% missing:

- `Ditë trajtimi`
- `Data limit`
- `Brenda afatit?`
- `Realizimi %` in the budget dataset

These are not necessarily data-quality errors because they can be recalculated from existing source fields.

For example:

- `Data limit` can be calculated from `Data e caktimit + Afati (ditë)`
- `Ditë trajtimi` can be calculated from `Data e zgjidhjes - Data e regjistrimit`
- `Brenda afatit?` can be calculated by comparing `Data e zgjidhjes` with `Data limit`
- `Realizimi %` can be calculated from `Shpenzimi real / Buxheti planifikuar`

These fields should therefore be rebuilt during preprocessing instead of filled manually.

#### 2. Workflow-related missing values

Some request fields are partially missing:

- `Data e zgjidhjes` is missing for 74 requests
- `Përgjegjësi`, `Drejtoria`, `Data e caktimit` and `Afati (ditë)` are missing for 28 requests
- `Shënim i përgjegjësit` is missing for 121 requests

These missing values may be valid depending on the request status.

For example, a new request may not yet be assigned to a directorate or responsible user, and an unresolved request should not yet have a resolution date.

The `Recurring group` field is missing for most requests because only a small number of requests are intentionally marked as recurring cases.

The operational datasets for waste and roads contain no missing values, which means they are structurally complete for further analysis.

## 5. Check duplicate records

Duplicate records can distort KPIs, trends and performance indicators.

We therefore check duplicates in two ways:

1. **Duplicate request IDs** — the same request ID should never appear more than once.
2. **Fully duplicated rows** — the exact same record should not be repeated.

For the operational and financial datasets, we also check whether complete duplicate rows exist.

In [ ]:
# 1. Duplicate request IDs
duplicate_request_ids = kerkesat_raw[
    kerkesat_raw.duplicated(subset=["ID"], keep=False)
]

print("Duplicate request IDs:", len(duplicate_request_ids))

if len(duplicate_request_ids) > 0:
    display(
        duplicate_request_ids.sort_values("ID")
    )


# 2. Fully duplicated rows in requests
duplicate_request_rows = kerkesat_raw[
    kerkesat_raw.duplicated(keep=False)
]

print("Fully duplicated request rows:", len(duplicate_request_rows))

if len(duplicate_request_rows) > 0:
    display(duplicate_request_rows)


# 3. Duplicates in other datasets
print(
    "Duplicate rows - Operacione_Mbetje:",
    mbetje_raw.duplicated().sum()
)

print(
    "Duplicate rows - Operacione_Rruge:",
    rruge_raw.duplicated().sum()
)

print(
    "Duplicate rows - Buxhet_Finance:",
    buxhet_raw.duplicated().sum()
)

Duplicate request IDs: 0
Fully duplicated request rows: 0
Duplicate rows - Operacione_Mbetje: 0
Duplicate rows - Operacione_Rruge: 0
Duplicate rows - Buxhet_Finance: 0


### Interpretation

No duplicate records were found in any of the datasets.

- `Kerkesat` contains no duplicate request IDs.
- No fully duplicated request rows were found.
- `Operacione_Mbetje`, `Operacione_Rruge`, and `Buxhet_Finance` also contain no duplicate rows.

This means that duplicate records will not distort the later KPI, trend, or performance calculations.

No duplicate-removal step is required at this stage.

## 6. Validate IDs and reference codes

After checking duplicates, we validate the identifiers used across the request dataset.

We verify that:

- every request has a valid `ID`;
- every `Njësia (kodi)` exists in the `Njesite` reference table;
- every assigned `Drejtoria` exists in the `Drejtorite` reference table.

This step helps us detect invalid or inconsistent references before moving to date and workflow validation.

In [ ]:
# Valid reference codes
valid_unit_codes = set(
    njesite["Kodi"]
    .astype(str)
    .str.strip()
)

valid_directorate_codes = set(
    drejtorite["Kodi"]
    .astype(str)
    .str.strip()
)

# 1. Missing request IDs
missing_request_ids = kerkesat_raw[
    kerkesat_raw["ID"].isna()
]

print("Missing request IDs:", len(missing_request_ids))


# 2. Invalid unit codes
invalid_unit_codes = kerkesat_raw[
    ~kerkesat_raw["Njësia (kodi)"].isin(valid_unit_codes)
]

print("Invalid unit codes:", len(invalid_unit_codes))

if len(invalid_unit_codes) > 0:
    display(
        invalid_unit_codes[
            ["ID", "Njësia (kodi)", "Njësia administrative"]
        ]
    )


# 3. Invalid directorate codes
invalid_directorates = kerkesat_raw[
    kerkesat_raw["Drejtoria"].notna()
    & ~kerkesat_raw["Drejtoria"].isin(valid_directorate_codes)
]

print("Invalid directorate codes:", len(invalid_directorates))

if len(invalid_directorates) > 0:
    display(
        invalid_directorates[
            ["ID", "Drejtoria", "Kategoria"]
        ]
    )

Missing request IDs: 0
Invalid unit codes: 0
Invalid directorate codes: 0


### Interpretation

All request identifiers and reference codes are valid.

- No request is missing an `ID`.
- Every `Njësia (kodi)` exists in the `Njesite` reference table.
- Every assigned `Drejtoria` exists in the `Drejtorite` reference table.

This means the request dataset is correctly linked to the reference structure and no ID correction is required.

## 7. Validate dates

Dates are critical for workflow analysis, SLA calculation and trend analysis.

We validate the request dates to make sure that:

- `Data e caktimit` is not earlier than `Data e regjistrimit`;
- `Data e zgjidhjes` is not earlier than `Data e regjistrimit`;
- unresolved requests do not contain a resolution date;
- resolved requests contain a valid resolution date;
- request registration dates belong to the expected analysis period.

These checks help ensure that later calculations such as resolution time and SLA performance are reliable.

In [ ]:
# Expected analysis period
start_date = pd.Timestamp("2026-06-01")
end_date = pd.Timestamp("2026-09-30")

# 1. Assignment date before registration date
assigned_before_created = kerkesat_raw[
    kerkesat_raw["Data e caktimit"].notna()
    & (
        kerkesat_raw["Data e caktimit"]
        < kerkesat_raw["Data e regjistrimit"]
    )
]

print(
    "Assignment date before registration date:",
    len(assigned_before_created)
)


# 2. Resolution date before registration date
resolved_before_created = kerkesat_raw[
    kerkesat_raw["Data e zgjidhjes"].notna()
    & (
        kerkesat_raw["Data e zgjidhjes"]
        < kerkesat_raw["Data e regjistrimit"]
    )
]

print(
    "Resolution date before registration date:",
    len(resolved_before_created)
)


# 3. Resolved requests without resolution date
resolved_without_date = kerkesat_raw[
    (kerkesat_raw["Statusi"] == "E zgjidhur")
    & kerkesat_raw["Data e zgjidhjes"].isna()
]

print(
    "Resolved requests without resolution date:",
    len(resolved_without_date)
)


# 4. Unresolved requests with resolution date
unresolved_with_date = kerkesat_raw[
    (kerkesat_raw["Statusi"] != "E zgjidhur")
    & kerkesat_raw["Data e zgjidhjes"].notna()
]

print(
    "Unresolved requests with resolution date:",
    len(unresolved_with_date)
)


# 5. Requests outside expected analysis period
outside_period = kerkesat_raw[
    (kerkesat_raw["Data e regjistrimit"] < start_date)
    | (kerkesat_raw["Data e regjistrimit"] > end_date)
]

print(
    "Requests outside expected period:",
    len(outside_period)
)

Assignment date before registration date: 0
Resolution date before registration date: 0
Resolved requests without resolution date: 0
Unresolved requests with resolution date: 0
Requests outside expected period: 0


### Interpretation

All date-related validation checks passed successfully.

- No request was assigned before it was registered.
- No request was resolved before it was registered.
- Every resolved request has a resolution date.
- No unresolved request contains a resolution date.
- All requests fall within the expected analysis period from June to September 2026.

This confirms that the request timeline is internally consistent and can be safely used for SLA and resolution-time calculations.

## 8. Validate request statuses

The request status is an important part of the municipal workflow.

We check that every request uses one of the expected status values:

- `E re`
- `E caktuar`
- `Në proces`
- `E ridrejtuar`
- `E zgjidhur`

We also inspect how many requests belong to each status.

This helps us verify that the workflow categories are consistent before we calculate open cases, resolution rates and SLA performance.

In [ ]:
valid_statuses = {
    "E re",
    "E caktuar",
    "Në proces",
    "E ridrejtuar",
    "E zgjidhur",
}

# 1. Invalid statuses
invalid_statuses = kerkesat_raw[
    ~kerkesat_raw["Statusi"].isin(valid_statuses)
]

print("Invalid statuses:", len(invalid_statuses))

if len(invalid_statuses) > 0:
    display(
        invalid_statuses[
            ["ID", "Statusi"]
        ]
    )


# 2. Status distribution
status_distribution = (
    kerkesat_raw["Statusi"]
    .value_counts()
    .to_frame("request_count")
)

status_distribution["percentage"] = (
    status_distribution["request_count"]
    / len(kerkesat_raw)
    * 100
).round(2)

display(status_distribution)

Invalid statuses: 0


,request_count,percentage
Statusi,,
E zgjidhur,482,86.69
E re,28,5.04
Në proces,23,4.14
E caktuar,17,3.06
E ridrejtuar,6,1.08


### Interpretation

All request status values are valid and consistent with the expected municipal workflow.

Most requests are already resolved:

- `E zgjidhur`: 482 requests (86.69%)
- `E re`: 28 requests (5.04%)
- `Në proces`: 23 requests (4.14%)
- `E caktuar`: 17 requests (3.06%)
- `E ridrejtuar`: 6 requests (1.08%)

This distribution shows that the dataset contains both completed and active cases.

The active requests (`E re`, `E caktuar`, `Në proces`, and `E ridrejtuar`) will later be important for calculating open-case KPIs, backlog, and SLA performance.

## 9. Validate category → directorate mapping

Each request category should be handled by the correct municipal directorate.

The `Kategorite` reference table defines this mapping.

Examples:

- `Mbetje të pagrumbulluara` → `DSHP`
- `Pastrim rrugësh` → `DSHP`
- `Gropa / rrugë e dëmtuar` → `DI`
- `Ndriçim publik` → `DI`
- `Kullim / ujëra atmosferike` → `DI`

We compare the directorate assigned in each request with the directorate expected from its category.

This helps us detect incorrectly routed requests.

In [ ]:
# Build category → directorate mapping from the reference table

category_to_directorate = (
    kategorite
    .set_index("Kategoria")["Drejtoria"]
    .to_dict()
)

# Expected directorate for each request
kerkesat_raw["Drejtoria_e_pritur"] = (
    kerkesat_raw["Kategoria"]
    .map(category_to_directorate)
)

# Find mismatches
category_directorate_mismatches = kerkesat_raw[
    kerkesat_raw["Drejtoria"].notna()
    & (
        kerkesat_raw["Drejtoria"]
        != kerkesat_raw["Drejtoria_e_pritur"]
    )
]

print(
    "Category → directorate mismatches:",
    len(category_directorate_mismatches)
)

if len(category_directorate_mismatches) > 0:
    display(
        category_directorate_mismatches[
            [
                "ID",
                "Kategoria",
                "Drejtoria",
                "Drejtoria_e_pritur",
            ]
        ]
    )

Category → directorate mismatches: 0


### Interpretation — Step 9

All request categories are assigned to the correct directorate.

No category → directorate mismatches were found.

This confirms that the routing logic between request categories and municipal directorates is consistent.

## 10. Validate assignee → directorate mapping

Each assigned request should be handled by a user who belongs to the same directorate as the request.

For example:

- requests assigned to `DSHP` should be handled by a DSHP responsible user;
- requests assigned to `DI` should be handled by a DI responsible user.

New requests may still have no assigned user, so missing assignees are not automatically considered errors.

This validation ensures that the request workflow is internally consistent.

In [ ]:
# Build user → directorate mapping

user_to_directorate = (
    perdoruesit
    .dropna(subset=["Drejtoria"])
    .set_index("Përdoruesi")["Drejtoria"]
    .to_dict()
)

# Expected directorate based on assignee
kerkesat_raw["Drejtoria_e_pergjegjesit"] = (
    kerkesat_raw["Përgjegjësi"]
    .map(user_to_directorate)
)

# Find mismatches
assignee_directorate_mismatches = kerkesat_raw[
    kerkesat_raw["Përgjegjësi"].notna()
    & kerkesat_raw["Drejtoria"].notna()
    & (
        kerkesat_raw["Drejtoria_e_pergjegjesit"]
        != kerkesat_raw["Drejtoria"]
    )
]

print(
    "Assignee → directorate mismatches:",
    len(assignee_directorate_mismatches)
)

if len(assignee_directorate_mismatches) > 0:
    display(
        assignee_directorate_mismatches[
            [
                "ID",
                "Drejtoria",
                "Përgjegjësi",
                "Drejtoria_e_pergjegjesit",
            ]
        ]
    )

Assignee → directorate mismatches: 0


### Interpretation — Step 10

All assigned requests are handled by users belonging to the correct directorate.

No assignee → directorate mismatches were found.

This confirms that the assignment workflow is internally consistent and that responsible users are correctly linked to their directorates.

## 11. Validate SLA and deadlines

Each request category has a predefined Service Level Agreement (SLA), expressed as a number of days.

The `Kategorite` reference table is the source of truth for these deadlines.

In this step we verify:

- whether the SLA stored in each request matches the expected SLA for its category;
- whether the deadline can be correctly calculated from the assignment date and SLA.

This is important because SLA performance will later be used to measure whether requests are resolved on time.

In [ ]:
# Build category → SLA mapping

category_to_sla = (
    kategorite
    .set_index("Kategoria")["Afati (ditë)"]
    .to_dict()
)

# Expected SLA for each request
kerkesat_raw["Afati_i_pritur"] = (
    kerkesat_raw["Kategoria"]
    .map(category_to_sla)
)

# Find SLA mismatches
sla_mismatches = kerkesat_raw[
    kerkesat_raw["Afati (ditë)"].notna()
    & (
        kerkesat_raw["Afati (ditë)"]
        != kerkesat_raw["Afati_i_pritur"]
    )
]

print(
    "SLA mismatches:",
    len(sla_mismatches)
)

if len(sla_mismatches) > 0:
    display(
        sla_mismatches[
            [
                "ID",
                "Kategoria",
                "Afati (ditë)",
                "Afati_i_pritur",
            ]
        ]
    )


# Recalculate expected deadline
kerkesat_raw["Data_limit_e_llogaritur"] = (
    kerkesat_raw["Data e caktimit"]
    + pd.to_timedelta(
        kerkesat_raw["Afati_i_pritur"],
        unit="D"
    )
)

display(
    kerkesat_raw[
        [
            "ID",
            "Kategoria",
            "Data e caktimit",
            "Afati_i_pritur",
            "Data_limit_e_llogaritur",
        ]
    ].head(10)
)

SLA mismatches: 0


,ID,Kategoria,Data e caktimit,Afati_i_pritur,Data_limit_e_llogaritur
0,K-2026-0526,Ndriçim publik,2026-06-02,5,2026-06-07
1,K-2026-0221,Gropa / rrugë e dëmtuar,2026-06-03,14,2026-06-17
2,K-2026-0229,Gropa / rrugë e dëmtuar,2026-06-02,14,2026-06-16
3,K-2026-0063,Ndriçim publik,2026-06-03,5,2026-06-08
4,K-2026-0107,Pemë / gjelbërim,2026-06-04,10,2026-06-14
5,K-2026-0222,Gropa / rrugë e dëmtuar,2026-06-03,14,2026-06-17
6,K-2026-0262,Mbetje të pagrumbulluara,2026-06-03,2,2026-06-05
7,K-2026-0525,Ndriçim publik,2026-06-03,5,2026-06-08
8,K-2026-0010,Mbetje të pagrumbulluara,2026-06-04,2,2026-06-06
9,K-2026-0314,Gropa / rrugë e dëmtuar,2026-06-05,14,2026-06-19


### Interpretation

All SLA values match the expected deadlines defined in the `Kategorite` reference table.

No SLA mismatches were found.

The request deadline can therefore be reliably reconstructed from:

`Data e caktimit + Afati_i_pritur`

The calculated deadlines are internally consistent and can be used later to evaluate SLA performance and determine whether resolved requests were completed on time.

## 12. Clean text fields

Text fields can contain small inconsistencies such as:

- extra spaces at the beginning or end;
- inconsistent capitalization;
- empty strings instead of proper missing values.

Even small text inconsistencies can create problems later when we group or compare categories, areas, users or directorates.

In this step, we standardize the main text fields without changing their actual meaning.

In [ ]:
# Create a working copy before cleaning
kerkesat_cleaning = kerkesat_raw.copy()

text_columns = [
    "ID",
    "Kanali",
    "Njësia (kodi)",
    "Njësia administrative",
    "Lagjja / Fshati",
    "Vendi",
    "Kategoria",
    "Përshkrimi",
    "Regjistruar nga",
    "Drejtoria",
    "Përgjegjësi",
    "Statusi",
    "Shënim i përgjegjësit",
    "Recurring group",
]

for column in text_columns:
    kerkesat_cleaning[column] = (
        kerkesat_cleaning[column]
        .astype("string")
        .str.strip()
    )

# Convert empty strings to missing values
kerkesat_cleaning[text_columns] = (
    kerkesat_cleaning[text_columns]
    .replace("", pd.NA)
)

print("Text cleaning completed.")

Text cleaning completed.


In [ ]:
for column in [
    "Kanali",
    "Njësia administrative",
    "Lagjja / Fshati",
    "Kategoria",
    "Drejtoria",
    "Statusi",
]:
    print(f"\n{column}:")
    print(sorted(
        kerkesat_cleaning[column]
        .dropna()
        .unique()
        .tolist()
    ))


Kanali:
['Email', 'Sportel', 'Telefon', 'Web']

Njësia administrative:
['Bradashesh', 'Elbasan (qyteti)', 'Papër']

Lagjja / Fshati:
['Bradashesh', 'Lagjja nr. 2', 'Lagjja nr. 3', 'Lagjja nr. 5', 'Papër']

Kategoria:
['Gropa / rrugë e dëmtuar', 'Kullim / ujëra atmosferike', 'Mbetje të pagrumbulluara', 'Ndriçim publik', 'Pastrim rrugësh', 'Pemë / gjelbërim']

Drejtoria:
['DI', 'DSHP']

Statusi:
['E caktuar', 'E re', 'E ridrejtuar', 'E zgjidhur', 'Në proces']


In [ ]:
### Interpretation

The main text fields are now standardized and internally consistent.

No duplicate text variants were found across channels, administrative units, areas, categories, directorates or statuses.

This means that grouping and comparison operations can be performed reliably without creating duplicate categories caused by formatting inconsistencies.

## 13. Recalculate derived fields

Some fields in the raw Excel file are derived from other columns and should not be treated as source-of-truth values.

We therefore recalculate them directly from the validated dates and SLA values.

We will create:

- `Data_limit_clean` — assignment date + SLA
- `Dite_trajtimi_clean` — resolution date - registration date
- `Brenda_afatit_clean` — whether a resolved request was completed before the deadline
- `Muaji` — month extracted from the registration date

These recalculated fields will be used later for KPI, SLA and trend analysis.

In [ ]:
# Work on the cleaned copy
kerkesat_cleaning["Data_limit_clean"] = (
    kerkesat_cleaning["Data e caktimit"]
    + pd.to_timedelta(
        kerkesat_cleaning["Afati_i_pritur"],
        unit="D"
    )
)

kerkesat_cleaning["Dite_trajtimi_clean"] = (
    kerkesat_cleaning["Data e zgjidhjes"]
    - kerkesat_cleaning["Data e regjistrimit"]
).dt.days

kerkesat_cleaning["Brenda_afatit_clean"] = pd.NA

resolved_mask = (
    (kerkesat_cleaning["Statusi"] == "E zgjidhur")
    & kerkesat_cleaning["Data e zgjidhjes"].notna()
    & kerkesat_cleaning["Data_limit_clean"].notna()
)

kerkesat_cleaning.loc[
    resolved_mask,
    "Brenda_afatit_clean"
] = (
    kerkesat_cleaning.loc[
        resolved_mask,
        "Data e zgjidhjes"
    ]
    <=
    kerkesat_cleaning.loc[
        resolved_mask,
        "Data_limit_clean"
    ]
)

kerkesat_cleaning["Muaji"] = (
    kerkesat_cleaning["Data e regjistrimit"]
    .dt.to_period("M")
)

display(
    kerkesat_cleaning[
        [
            "ID",
            "Data e regjistrimit",
            "Data e caktimit",
            "Afati_i_pritur",
            "Data_limit_clean",
            "Data e zgjidhjes",
            "Dite_trajtimi_clean",
            "Brenda_afatit_clean",
            "Muaji",
        ]
    ].head(10)
)

,ID,Data e regjistrimit,Data e caktimit,Afati_i_pritur,Data_limit_clean,Data e zgjidhjes,Dite_trajtimi_clean,Brenda_afatit_clean,Muaji
0,K-2026-0526,2026-06-01,2026-06-02,5,2026-06-07,2026-06-05,4.0,True,2026-06
1,K-2026-0221,2026-06-02,2026-06-03,14,2026-06-17,2026-06-04,2.0,True,2026-06
2,K-2026-0229,2026-06-02,2026-06-02,14,2026-06-16,2026-06-15,13.0,True,2026-06
3,K-2026-0063,2026-06-03,2026-06-03,5,2026-06-08,2026-06-08,5.0,True,2026-06
4,K-2026-0107,2026-06-03,2026-06-04,10,2026-06-14,2026-06-11,8.0,True,2026-06
5,K-2026-0222,2026-06-03,2026-06-03,14,2026-06-17,2026-06-11,8.0,True,2026-06
6,K-2026-0262,2026-06-03,2026-06-03,2,2026-06-05,2026-06-05,2.0,True,2026-06
7,K-2026-0525,2026-06-03,2026-06-03,5,2026-06-08,2026-06-08,5.0,True,2026-06
8,K-2026-0010,2026-06-04,2026-06-04,2,2026-06-06,2026-06-05,1.0,True,2026-06
9,K-2026-0314,2026-06-04,2026-06-05,14,2026-06-19,2026-06-18,14.0,True,2026-06


### Interpretation

The derived fields were successfully recalculated from the validated source data.

- `Data_limit_clean` represents the expected deadline based on assignment date and SLA.
- `Dite_trajtimi_clean` measures the actual number of days required to resolve a request.
- `Brenda_afatit_clean` identifies whether a resolved request was completed within its deadline.
- `Muaji` provides a standardized monthly field for trend analysis.

These recalculated values will be used as the source of truth for later KPI, SLA and trend calculations instead of relying on the empty derived fields from the original Excel file.

sla_result = (
    kerkesat_cleaning.loc[
        kerkesat_cleaning["Statusi"] == "E zgjidhur",
        "Brenda_afatit_clean"
    ]
    .value_counts()
)

display(sla_result)

sla_pct = (
    sla_result
    / sla_result.sum()
    * 100
).round(2)

display(sla_pct)

### Interpretation

Among the 482 resolved requests:

- 421 requests (87.34%) were completed within the expected SLA.
- 61 requests (12.66%) were completed after the deadline.

This shows that the municipality performs well overall, but a meaningful share of resolved requests still exceed their expected service deadlines.

The SLA result is therefore realistic enough to support later comparisons by area, directorate and month.

## 14. Detect recurring issues

A resolved request does not always mean that the underlying problem has been permanently solved.

Some issues may be reported again in the same area or location after an earlier case was closed.

The `Recurring group` field helps us identify these repeated cases.

In this step, we group recurring requests and measure:

- how many times the issue was reported;
- where it happened;
- which category it belongs to;
- when it was first reported;
- when it was last reported;
- how many of the related requests were resolved.

This creates the basis for the **Closed ≠ Solved** insight.

In [ ]:
# Mark recurring requests
kerkesat_cleaning["Is_recurring"] = (
    kerkesat_cleaning["Recurring group"]
    .notna()
)

# Summary of recurring issues
recurring_summary = (
    kerkesat_cleaning[
        kerkesat_cleaning["Is_recurring"]
    ]
    .groupby(
        [
            "Recurring group",
            "Njësia administrative",
            "Lagjja / Fshati",
            "Vendi",
            "Kategoria",
        ],
        dropna=False,
    )
    .agg(
        Numri_raportimeve=("ID", "count"),
        Raportimi_i_pare=("Data e regjistrimit", "min"),
        Raportimi_i_fundit=("Data e regjistrimit", "max"),
        Te_zgjidhura=(
            "Statusi",
            lambda x: (x == "E zgjidhur").sum()
        ),
    )
    .reset_index()
    .sort_values(
        "Numri_raportimeve",
        ascending=False
    )
)

display(recurring_summary)

,Recurring group,Njësia administrative,Lagjja / Fshati,Vendi,Kategoria,Numri_raportimeve,Raportimi_i_pare,Raportimi_i_fundit,Te_zgjidhura
0,REC-001,Elbasan (qyteti),Lagjja nr. 3,Tregu i lagjes,Mbetje të pagrumbulluara,6,2026-06-24,2026-09-27,6
1,REC-002,Elbasan (qyteti),Lagjja nr. 3,Pallati nr. 12,Mbetje të pagrumbulluara,6,2026-06-25,2026-09-29,5
2,REC-003,Elbasan (qyteti),Lagjja nr. 5,Kryqëzimi kryesor,Kullim / ujëra atmosferike,6,2026-06-08,2026-09-26,5
3,REC-004,Bradashesh,Bradashesh,Rruga kryesore,Gropa / rrugë e dëmtuar,6,2026-08-11,2026-09-30,5


In [ ]:
total_recurring_requests = (
    kerkesat_cleaning["Is_recurring"]
    .sum()
)

unique_recurring_groups = (
    kerkesat_cleaning["Recurring group"]
    .nunique()
)

print(
    "Recurring requests:",
    total_recurring_requests
)

print(
    "Recurring issue groups:",
    unique_recurring_groups
)

Recurring requests: 24
Recurring issue groups: 4


### Interpretation

The recurring-issue analysis identified 24 repeated requests grouped into 4 recurring issue groups.

Each recurring issue was reported 6 times across the analysis period.

Several of these groups contain requests that were marked as resolved, but the same issue was reported again later at the same location.

This supports the `Closed ≠ Solved` concept: administrative closure does not always mean that the underlying problem has been permanently resolved.

These recurring groups will later be used as an important decision-support signal in Luma.

## 15. Clean waste operations

The waste-operations dataset contains the operational indicators used to evaluate the performance of the Public Services Directorate (DSHP).

In this step we:

- standardize column names;
- convert the date field to datetime;
- normalize text fields;
- create a monthly field for later trend analysis;
- validate numeric indicators.

The main operational checks are:

- planned collections must be non-negative;
- completed collections cannot exceed planned collections;
- missed collections cannot exceed planned collections;
- average delay cannot be negative;
- vehicle availability must remain between 0% and 100%.

These checks ensure that the operational data can be safely used for cross-source analysis later.

In [ ]:
# Create a working copy
mbetje_cleaning = mbetje_raw.copy()

# Rename columns
mbetje_cleaning = mbetje_cleaning.rename(
    columns={
        "Data": "date",
        "Njësia (kodi)": "unit_code",
        "Njësia administrative": "administrative_unit",
        "Lagjja / Fshati": "area",
        "Drejtoria": "directorate",
        "Mbledhje planifikuara": "scheduled_collections",
        "Mbledhje realizuara": "completed_collections",
        "Mbledhje të humbura": "missed_collections",
        "Vonesa mesatare (orë)": "avg_delay_hours",
        "Disponueshmëri mjetesh %": "vehicle_availability_pct",
    }
)

# Convert date
mbetje_cleaning["date"] = pd.to_datetime(
    mbetje_cleaning["date"],
    errors="coerce"
)

# Clean text fields
for column in [
    "unit_code",
    "administrative_unit",
    "area",
    "directorate",
]:
    mbetje_cleaning[column] = (
        mbetje_cleaning[column]
        .astype("string")
        .str.strip()
    )

# Create month field
mbetje_cleaning["month"] = (
    mbetje_cleaning["date"]
    .dt.to_period("M")
)

display(mbetje_cleaning.head())

,date,unit_code,administrative_unit,area,directorate,scheduled_collections,completed_collections,missed_collections,avg_delay_hours,vehicle_availability_pct,month
0,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DSHP,111,104,7,1.2,96.8,2026-06
1,2026-06-08,NJ01,Elbasan (qyteti),Lagjja nr. 2,DSHP,131,130,1,0.5,88.6,2026-06
2,2026-06-15,NJ01,Elbasan (qyteti),Lagjja nr. 2,DSHP,126,119,7,0.7,91.3,2026-06
3,2026-06-22,NJ01,Elbasan (qyteti),Lagjja nr. 2,DSHP,109,106,3,1.9,95.6,2026-06
4,2026-07-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DSHP,100,97,3,1.9,92.1,2026-07


In [ ]:
waste_quality_checks = {
    "Missing dates":
        mbetje_cleaning["date"].isna().sum(),

    "Negative planned collections":
        (mbetje_cleaning["scheduled_collections"] < 0).sum(),

    "Negative completed collections":
        (mbetje_cleaning["completed_collections"] < 0).sum(),

    "Negative missed collections":
        (mbetje_cleaning["missed_collections"] < 0).sum(),

    "Completed > planned":
        (
            mbetje_cleaning["completed_collections"]
            > mbetje_cleaning["scheduled_collections"]
        ).sum(),

    "Missed > planned":
        (
            mbetje_cleaning["missed_collections"]
            > mbetje_cleaning["scheduled_collections"]
        ).sum(),

    "Negative delay":
        (mbetje_cleaning["avg_delay_hours"] < 0).sum(),

    "Vehicle availability outside 0–100":
        (
            (mbetje_cleaning["vehicle_availability_pct"] < 0)
            |
            (mbetje_cleaning["vehicle_availability_pct"] > 100)
        ).sum(),
}

for check, result in waste_quality_checks.items():
    print(f"{check}: {result}")

Missing dates: 0
Negative planned collections: 0
Negative completed collections: 0
Negative missed collections: 0
Completed > planned: 0
Missed > planned: 0
Negative delay: 0
Vehicle availability outside 0–100: 0


### Interpretation

The waste-operations dataset passed all validation checks.

- No dates are missing.
- No negative collection values were found.
- Completed collections never exceed planned collections.
- Missed collections never exceed planned collections.
- Average delay values are valid.
- Vehicle availability remains within the expected 0–100% range.

The dataset is therefore structurally consistent and ready for trend and cross-source analysis.

The new `month` field will later allow us to compare waste-service performance across June, July, August and September 2026.

In [ ]:
# Create a working copy
rruge_cleaning = rruge_raw.copy()

# Rename columns
rruge_cleaning = rruge_cleaning.rename(
    columns={
        "Muaji": "month_date",
        "Njësia (kodi)": "unit_code",
        "Njësia administrative": "administrative_unit",
        "Lagjja / Fshati": "area",
        "Drejtoria": "directorate",
        "Incidente të raportuara": "reported_incidents",
        "Incidente të hapura": "open_incidents",
        "Ditë mesatare riparimi": "avg_repair_days",
        "Raste kullimi": "drainage_cases",
        "Ndërhyrje urgjente": "emergency_repairs",
    }
)

# Convert month/date
rruge_cleaning["month_date"] = pd.to_datetime(
    rruge_cleaning["month_date"],
    errors="coerce"
)

# Clean text fields
for column in [
    "unit_code",
    "administrative_unit",
    "area",
    "directorate",
]:
    rruge_cleaning[column] = (
        rruge_cleaning[column]
        .astype("string")
        .str.strip()
    )

# Create month field
rruge_cleaning["month"] = (
    rruge_cleaning["month_date"]
    .dt.to_period("M")
)

display(rruge_cleaning.head())

,month_date,unit_code,administrative_unit,area,directorate,reported_incidents,open_incidents,avg_repair_days,drainage_cases,emergency_repairs,month
0,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DI,12,1,4.0,1,3,2026-06
1,2026-07-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DI,13,4,4.6,4,2,2026-07
2,2026-08-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DI,7,4,4.0,1,2,2026-08
3,2026-09-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DI,14,3,3.2,4,0,2026-09
4,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 3,DI,14,2,4.1,2,1,2026-06


In [ ]:
road_quality_checks = {
    "Missing month dates":
        rruge_cleaning["month_date"].isna().sum(),

    "Negative reported incidents":
        (rruge_cleaning["reported_incidents"] < 0).sum(),

    "Negative open incidents":
        (rruge_cleaning["open_incidents"] < 0).sum(),

    "Open incidents > reported incidents":
        (
            rruge_cleaning["open_incidents"]
            > rruge_cleaning["reported_incidents"]
        ).sum(),

    "Negative repair time":
        (rruge_cleaning["avg_repair_days"] < 0).sum(),

    "Negative drainage cases":
        (rruge_cleaning["drainage_cases"] < 0).sum(),

    "Negative emergency repairs":
        (rruge_cleaning["emergency_repairs"] < 0).sum(),
}

for check, result in road_quality_checks.items():
    print(f"{check}: {result}")

Missing month dates: 0
Negative reported incidents: 0
Negative open incidents: 0
Open incidents > reported incidents: 0
Negative repair time: 0
Negative drainage cases: 0
Negative emergency repairs: 0


### Interpretation

The road and infrastructure operations dataset passed all validation checks.

- No monthly dates are missing.
- No negative incident values were found.
- Open incidents never exceed reported incidents.
- Average repair time values are valid.
- Drainage cases and emergency interventions are non-negative.

The dataset is therefore structurally consistent and ready for trend, KPI and cross-source analysis.

The monthly field created during cleaning will allow infrastructure performance to be compared across June, July, August and September 2026.

## 17. Clean budget data

The budget dataset provides the financial context for DSHP and DI.

In this step we:

- standardize column names;
- convert the month field to datetime;
- clean text fields;
- recalculate budget execution percentage;
- validate that financial values are logically consistent.

The main checks are:

- planned budget must be greater than zero;
- actual spending cannot be negative;
- committed spending cannot be negative;
- budget execution must be calculated from actual spending / planned budget.

In [ ]:
# Create a working copy
buxhet_cleaning = buxhet_raw.copy()

# Rename columns
buxhet_cleaning = buxhet_cleaning.rename(
    columns={
        "Muaji": "month_date",
        "Kodi drejtoria": "directorate",
        "Drejtoria": "directorate_name",
        "Buxheti planifikuar ALL": "budget_allocated_all",
        "Shpenzimi real ALL": "actual_spend_all",
        "Shpenzimi angazhuar ALL": "committed_spend_all",
        "Realizimi %": "execution_pct_excel",
    }
)

# Convert month
buxhet_cleaning["month_date"] = pd.to_datetime(
    buxhet_cleaning["month_date"],
    errors="coerce"
)

# Clean text
for column in ["directorate", "directorate_name"]:
    buxhet_cleaning[column] = (
        buxhet_cleaning[column]
        .astype("string")
        .str.strip()
    )

# Create month field
buxhet_cleaning["month"] = (
    buxhet_cleaning["month_date"]
    .dt.to_period("M")
)

# Recalculate execution percentage
buxhet_cleaning["execution_pct"] = (
    buxhet_cleaning["actual_spend_all"]
    / buxhet_cleaning["budget_allocated_all"]
    * 100
).round(2)

display(buxhet_cleaning)

,month_date,directorate,directorate_name,budget_allocated_all,actual_spend_all,committed_spend_all,execution_pct_excel,month,execution_pct
0,2026-06-01,DSHP,Drejtoria e Shërbimeve Publike,41000000,32900000,35780879,NaN,2026-06,80.24
1,2026-06-01,DI,Drejtoria e Infrastrukturës,50000000,35900000,37181840,NaN,2026-06,71.80
2,2026-07-01,DSHP,Drejtoria e Shërbimeve Publike,42000000,34100000,37141904,NaN,2026-07,81.19
3,2026-07-01,DI,Drejtoria e Infrastrukturës,51000000,37100000,40486799,NaN,2026-07,72.75
4,2026-08-01,DSHP,Drejtoria e Shërbimeve Publike,45000000,41600000,45745565,NaN,2026-08,92.44
5,2026-08-01,DI,Drejtoria e Infrastrukturës,53500000,40200000,44004099,NaN,2026-08,75.14
6,2026-09-01,DSHP,Drejtoria e Shërbimeve Publike,48000000,49100000,51379073,NaN,2026-09,102.29
7,2026-09-01,DI,Drejtoria e Infrastrukturës,56000000,43100000,45900762,NaN,2026-09,76.96


In [ ]:
budget_quality_checks = {
    "Missing month dates":
        buxhet_cleaning["month_date"].isna().sum(),

    "Budget <= 0":
        (buxhet_cleaning["budget_allocated_all"] <= 0).sum(),

    "Negative actual spend":
        (buxhet_cleaning["actual_spend_all"] < 0).sum(),

    "Negative committed spend":
        (buxhet_cleaning["committed_spend_all"] < 0).sum(),
}

for check, result in budget_quality_checks.items():
    print(f"{check}: {result}")

display(
    buxhet_cleaning[
        [
            "month",
            "directorate",
            "budget_allocated_all",
            "actual_spend_all",
            "execution_pct"
        ]
    ]
)

Missing month dates: 0
Budget <= 0: 0
Negative actual spend: 0
Negative committed spend: 0


,month,directorate,budget_allocated_all,actual_spend_all,execution_pct
0,2026-06,DSHP,41000000,32900000,80.24
1,2026-06,DI,50000000,35900000,71.80
2,2026-07,DSHP,42000000,34100000,81.19
3,2026-07,DI,51000000,37100000,72.75
4,2026-08,DSHP,45000000,41600000,92.44
5,2026-08,DI,53500000,40200000,75.14
6,2026-09,DSHP,48000000,49100000,102.29
7,2026-09,DI,56000000,43100000,76.96


### Interpretation

The budget dataset passed all structural validation checks.

- No month values are missing.
- Planned budgets are positive.
- Actual and committed spending values are non-negative.

The recalculated budget execution percentages show different spending patterns across the two directorates.

For DSHP, budget execution increased from 80.24% in June to 102.29% in September.

For DI, budget execution increased more gradually, from 71.80% in June to 76.96% in September.

The September DSHP result is particularly important because spending exceeded the monthly planned budget. This will later be compared with service demand and operational performance to understand whether financial pressure coincides with service deterioration.

## 18. Final data-quality report

After validating the request workflow, operational datasets and budget data, we combine the main validation results into one final data-quality report.

The purpose of this step is to summarize whether the datasets are reliable enough to continue with analytics.

A `PASS` result means that no issue was detected for that validation check.

A `REVIEW` result means that the issue should be inspected before the data is used for KPI or decision-support calculations.

In [ ]:
quality_report = pd.DataFrame(
    [
        ["Duplicate request IDs", len(duplicate_request_ids)],
        ["Duplicate request rows", len(duplicate_request_rows)],
        ["Invalid unit codes", len(invalid_unit_codes)],
        ["Invalid directorate codes", len(invalid_directorates)],
        ["Assignment before registration", len(assigned_before_created)],
        ["Resolution before registration", len(resolved_before_created)],
        ["Resolved without resolution date", len(resolved_without_date)],
        ["Unresolved with resolution date", len(unresolved_with_date)],
        ["Requests outside analysis period", len(outside_period)],
        ["Invalid request statuses", len(invalid_statuses)],
        ["Category → directorate mismatches", len(category_directorate_mismatches)],
        ["Assignee → directorate mismatches", len(assignee_directorate_mismatches)],
        ["SLA mismatches", len(sla_mismatches)],
        ["Waste data-quality issues", sum(waste_quality_checks.values())],
        ["Road data-quality issues", sum(road_quality_checks.values())],
        ["Budget data-quality issues", sum(budget_quality_checks.values())],
    ],
    columns=["Validation check", "Issue count"],
)

quality_report["Status"] = np.where(
    quality_report["Issue count"] == 0,
    "PASS",
    "REVIEW",
)

display(quality_report)

,Validation check,Issue count,Status
0,Duplicate request IDs,0,PASS
1,Duplicate request rows,0,PASS
2,Invalid unit codes,0,PASS
3,Invalid directorate codes,0,PASS
4,Assignment before registration,0,PASS
5,Resolution before registration,0,PASS
6,Resolved without resolution date,0,PASS
7,Unresolved with resolution date,0,PASS
8,Requests outside analysis period,0,PASS
9,Invalid request statuses,0,PASS


In [ ]:
total_checks = len(quality_report)

passed_checks = (
    quality_report["Status"] == "PASS"
).sum()

review_checks = (
    quality_report["Status"] == "REVIEW"
).sum()

print("Total validation checks:", total_checks)
print("Passed checks:", passed_checks)
print("Checks requiring review:", review_checks)

Total validation checks: 16
Passed checks: 16
Checks requiring review: 0


### Interpretation

All 16 validation checks passed successfully.

No duplicate records, invalid IDs, inconsistent dates, routing errors, SLA mismatches, or operational data-quality issues were detected.

This confirms that the dataset is structurally consistent and reliable enough to move into the analytics stage.

Although the raw data is clean, the preprocessing phase was still necessary because it validated the business logic, rebuilt derived fields, and prepared trustworthy inputs for KPI, trend, SLA, recurring-issue and cross-source analysis.

## 19. Create clean datasets

After completing the validation and preprocessing steps, we now create the final clean datasets that will be used for analytics.

The goal is to keep only the validated and useful fields and separate the cleaned data from the original raw Excel input.

The final datasets will be:

- `kerkesat_clean`
- `mbetje_clean`
- `rruge_clean`
- `buxhet_clean`

These datasets will become the trusted inputs for KPI calculation, trend analysis, SLA performance, recurring-issue detection and cross-source decision support.

In [ ]:
# Final clean citizen-request dataset
kerkesat_clean = kerkesat_cleaning[
    [
        "ID",
        "Data e regjistrimit",
        "Kanali",
        "Njësia (kodi)",
        "Njësia administrative",
        "Lagjja / Fshati",
        "Vendi",
        "Kategoria",
        "Përshkrimi",
        "Regjistruar nga",
        "Drejtoria",
        "Përgjegjësi",
        "Data e caktimit",
        "Afati_i_pritur",
        "Data_limit_clean",
        "Statusi",
        "Data e zgjidhjes",
        "Shënim i përgjegjësit",
        "Dite_trajtimi_clean",
        "Brenda_afatit_clean",
        "Recurring group",
        "Is_recurring",
        "Muaji",
    ]
].copy()


# Final clean waste-operations dataset
mbetje_clean = mbetje_cleaning.copy()


# Final clean road-operations dataset
rruge_clean = rruge_cleaning.copy()


# Final clean budget dataset
buxhet_clean = buxhet_cleaning[
    [
        "month_date",
        "month",
        "directorate",
        "directorate_name",
        "budget_allocated_all",
        "actual_spend_all",
        "committed_spend_all",
        "execution_pct",
    ]
].copy()


# Sort datasets
kerkesat_clean = kerkesat_clean.sort_values(
    ["Data e regjistrimit", "ID"]
).reset_index(drop=True)

mbetje_clean = mbetje_clean.sort_values(
    ["date", "administrative_unit", "area"]
).reset_index(drop=True)

rruge_clean = rruge_clean.sort_values(
    ["month_date", "administrative_unit", "area"]
).reset_index(drop=True)

buxhet_clean = buxhet_clean.sort_values(
    ["month_date", "directorate"]
).reset_index(drop=True)


print("Kerkesat clean:", kerkesat_clean.shape)
print("Mbetje clean:", mbetje_clean.shape)
print("Rruge clean:", rruge_clean.shape)
print("Buxhet clean:", buxhet_clean.shape)

Kerkesat clean: (556, 23)
Mbetje clean: (80, 11)
Rruge clean: (20, 11)
Buxhet clean: (8, 8)


In [ ]:
display(kerkesat_clean.head())
display(mbetje_clean.head())
display(rruge_clean.head())
display(buxhet_clean.head())

,ID,Data e regjistrimit,Kanali,Njësia (kodi),Njësia administrative,Lagjja / Fshati,Vendi,Kategoria,Përshkrimi,Regjistruar nga,...,Afati_i_pritur,Data_limit_clean,Statusi,Data e zgjidhjes,Shënim i përgjegjësit,Dite_trajtimi_clean,Brenda_afatit_clean,Recurring group,Is_recurring,Muaji
0,K-2026-0526,2026-06-01,Telefon,NJ09,Papër,Papër,Stacioni,Ndriçim publik,Ndriçimi mungon pranë shkollës.,operator.telefon,...,5,2026-06-07,E zgjidhur,2026-06-05,<NA>,4.0,True,<NA>,False,2026-06
1,K-2026-0221,2026-06-02,Email,NJ01,Elbasan (qyteti),Lagjja nr. 3,Parku qendror,Gropa / rrugë e dëmtuar,Trotuari është dëmtuar dhe është i rrezikshëm ...,operator.sporteli,...,14,2026-06-17,E zgjidhur,2026-06-04,Ndërhyrja u krye dhe rasti u mbyll.,2.0,True,<NA>,False,2026-06
2,K-2026-0229,2026-06-02,Telefon,NJ01,Elbasan (qyteti),Lagjja nr. 3,Pallati nr. 12,Gropa / rrugë e dëmtuar,Gropë e madhe po pengon qarkullimin.,operator.telefon,...,14,2026-06-16,E zgjidhur,2026-06-15,Ndërhyrja u krye dhe rasti u mbyll.,13.0,True,<NA>,False,2026-06
3,K-2026-0063,2026-06-03,Telefon,NJ01,Elbasan (qyteti),Lagjja nr. 2,Tregu i lagjes,Ndriçim publik,Ndriçimi mungon pranë shkollës.,operator.telefon,...,5,2026-06-08,E zgjidhur,2026-06-08,Ndërhyrja u krye dhe rasti u mbyll.,5.0,True,<NA>,False,2026-06
4,K-2026-0107,2026-06-03,Telefon,NJ01,Elbasan (qyteti),Lagjja nr. 3,Pallati nr. 12,Pemë / gjelbërim,Degë të rënda rrezikojnë kalimtarët.,operator.telefon,...,10,2026-06-14,E zgjidhur,2026-06-11,<NA>,8.0,True,<NA>,False,2026-06


,date,unit_code,administrative_unit,area,directorate,scheduled_collections,completed_collections,missed_collections,avg_delay_hours,vehicle_availability_pct,month
0,2026-06-01,NJ02,Bradashesh,Bradashesh,DSHP,102,100,2,0.8,89.5,2026-06
1,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DSHP,111,104,7,1.2,96.8,2026-06
2,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 3,DSHP,139,135,4,1.4,89.0,2026-06
3,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 5,DSHP,102,99,3,1.4,90.7,2026-06
4,2026-06-01,NJ09,Papër,Papër,DSHP,125,118,7,0.6,89.6,2026-06


,month_date,unit_code,administrative_unit,area,directorate,reported_incidents,open_incidents,avg_repair_days,drainage_cases,emergency_repairs,month
0,2026-06-01,NJ02,Bradashesh,Bradashesh,DI,11,1,5.2,3,3,2026-06
1,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 2,DI,12,1,4.0,1,3,2026-06
2,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 3,DI,14,2,4.1,2,1,2026-06
3,2026-06-01,NJ01,Elbasan (qyteti),Lagjja nr. 5,DI,13,4,4.5,2,3,2026-06
4,2026-06-01,NJ09,Papër,Papër,DI,10,2,3.7,4,3,2026-06


,month_date,month,directorate,directorate_name,budget_allocated_all,actual_spend_all,committed_spend_all,execution_pct
0,2026-06-01,2026-06,DI,Drejtoria e Infrastrukturës,50000000,35900000,37181840,71.80
1,2026-06-01,2026-06,DSHP,Drejtoria e Shërbimeve Publike,41000000,32900000,35780879,80.24
2,2026-07-01,2026-07,DI,Drejtoria e Infrastrukturës,51000000,37100000,40486799,72.75
3,2026-07-01,2026-07,DSHP,Drejtoria e Shërbimeve Publike,42000000,34100000,37141904,81.19
4,2026-08-01,2026-08,DI,Drejtoria e Infrastrukturës,53500000,40200000,44004099,75.14


### Interpretation

The preprocessing phase is now complete.

Four validated and cleaned datasets have been created:

- `kerkesat_clean` contains the complete cleaned citizen-request workflow;
- `mbetje_clean` contains validated waste-service operations;
- `rruge_clean` contains validated road and infrastructure operations;
- `buxhet_clean` contains recalculated and validated financial indicators.

The original raw datasets remain unchanged, while these clean datasets will now serve as the source of truth for the analytics and decision-support stages.

## 20. Analytics overview

Now that the preprocessing phase is complete, we can start analyzing the cleaned datasets.

Before calculating final KPIs, we first create a general overview of the request data.

We will examine:

- request volume by month;
- request volume by area;
- request volume by directorate.

This step helps us understand the overall distribution of municipal demand before moving into more specific performance indicators.

In [ ]:
# Requests by month
requests_by_month = (
    kerkesat_clean
    .groupby("Muaji")
    .size()
    .reset_index(name="requests")
)

display(requests_by_month)


# Requests by area
requests_by_area = (
    kerkesat_clean
    .groupby("Lagjja / Fshati")
    .size()
    .reset_index(name="requests")
    .sort_values("requests", ascending=False)
)

display(requests_by_area)


# Requests by directorate
requests_by_directorate = (
    kerkesat_clean
    .groupby("Drejtoria", dropna=False)
    .size()
    .reset_index(name="requests")
    .sort_values("requests", ascending=False)
)

display(requests_by_directorate)

,Muaji,requests
0,2026-06,107
1,2026-07,121
2,2026-08,136
3,2026-09,192


,Lagjja / Fshati,requests
2,Lagjja nr. 3,162
3,Lagjja nr. 5,114
0,Bradashesh,102
1,Lagjja nr. 2,98
4,Papër,80


,Drejtoria,requests
1,DSHP,305
0,DI,223
2,<NA>,28


### Interpretation

The analytics overview provides the first high-level view of municipal demand.

The monthly request totals show whether citizen demand is increasing, decreasing or remaining stable over time.

The area-level distribution highlights which neighbourhoods or administrative units generate the most requests.

The directorate-level distribution shows how the workload is divided between DSHP and DI.

These results provide the baseline for the next steps, where we will calculate performance KPIs rather than only request volumes.

## 20. Analytics overview

Now that preprocessing is complete, we begin the analytics stage.

We first create a high-level view of municipal demand by:

- month;
- area;
- directorate.

This gives us the baseline before calculating performance KPIs.

In [ ]:
# Requests by month
requests_by_month = (
    kerkesat_clean
    .groupby("Muaji")
    .size()
    .reset_index(name="requests")
)

# Requests by area
requests_by_area = (
    kerkesat_clean
    .groupby("Lagjja / Fshati")
    .size()
    .reset_index(name="requests")
    .sort_values("requests", ascending=False)
)

# Requests by directorate
requests_by_directorate = (
    kerkesat_clean
    .groupby("Drejtoria", dropna=False)
    .size()
    .reset_index(name="requests")
    .sort_values("requests", ascending=False)
)

display(requests_by_month)
display(requests_by_area)
display(requests_by_directorate)

,Muaji,requests
0,2026-06,107
1,2026-07,121
2,2026-08,136
3,2026-09,192


,Lagjja / Fshati,requests
2,Lagjja nr. 3,162
3,Lagjja nr. 5,114
0,Bradashesh,102
1,Lagjja nr. 2,98
4,Papër,80


,Drejtoria,requests
1,DSHP,305
0,DI,223
2,<NA>,28


### Interpretation

The overall request volume increased steadily from June to September 2026:

- June: 107 requests
- July: 121 requests
- August: 136 requests
- September: 192 requests

This indicates a clear increase in municipal service demand, with the strongest rise occurring in September.

At area level, `Lagjja nr. 3` generated the highest number of requests with 162, followed by `Lagjja nr. 5` with 114.

At directorate level:

- DSHP handled 305 requests;
- DI handled 223 requests;
- 28 requests were still unassigned.

The unassigned requests correspond to cases that are still new and have not yet been routed to a directorate.

This overview suggests that later analysis should pay particular attention to September, Lagjja nr. 3, and the workload of DSHP.

## 21. Calculate core KPIs

The next step is to calculate the main performance indicators for the municipal request-management system.

The core KPIs are:

- total requests;
- active requests;
- resolved requests;
- resolution rate;
- average resolution time;
- SLA compliance rate;
- recurring requests.

These indicators summarize the overall performance of the request-management system and will later be shown in the dashboard.

In [ ]:
# Total number of requests
total_requests = len(kerkesat_clean)

# Resolved requests
resolved_requests = (
    kerkesat_clean["Statusi"] == "E zgjidhur"
).sum()

# Active requests
active_requests = (
    kerkesat_clean["Statusi"]
    .isin([
        "E re",
        "E caktuar",
        "Në proces",
        "E ridrejtuar"
    ])
).sum()

# Resolution rate
resolution_rate = (
    resolved_requests
    / total_requests
    * 100
)

# Average resolution time
avg_resolution_days = (
    kerkesat_clean.loc[
        kerkesat_clean["Statusi"] == "E zgjidhur",
        "Dite_trajtimi_clean"
    ]
    .mean()
)

# SLA compliance rate
sla_compliance_rate = (
    kerkesat_clean.loc[
        kerkesat_clean["Statusi"] == "E zgjidhur",
        "Brenda_afatit_clean"
    ]
    .mean()
    * 100
)

# Recurring requests
recurring_requests = (
    kerkesat_clean["Is_recurring"]
    .sum()
)

# KPI summary table
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Requests",
        "Active Requests",
        "Resolved Requests",
        "Resolution Rate (%)",
        "Average Resolution Time (days)",
        "SLA Compliance (%)",
        "Recurring Requests",
    ],
    "Value": [
        total_requests,
        active_requests,
        resolved_requests,
        round(resolution_rate, 2),
        round(avg_resolution_days, 2),
        round(sla_compliance_rate, 2),
        recurring_requests,
    ],
})

display(kpi_summary)

,KPI,Value
0,Total Requests,556.00
1,Active Requests,74.00
2,Resolved Requests,482.00
3,Resolution Rate (%),86.69
4,Average Resolution Time (days),4.90
5,SLA Compliance (%),87.34
6,Recurring Requests,24.00


### Interpretation

The core KPIs show that the municipal request-management system is performing relatively well overall, while still containing clear areas for improvement.

During the June–September 2026 period:

- 556 requests were registered.
- 482 requests were resolved.
- 74 requests remain active.
- The overall resolution rate is 86.69%.
- Resolved requests required an average of 4.90 days.
- 87.34% of resolved requests were completed within their SLA.
- 24 requests belong to recurring issue groups.

These results show that high overall resolution does not mean that all service problems are fully solved. Active cases, SLA violations and recurring requests remain important signals for management attention.

## 22. Analyze monthly trends

After calculating the overall KPIs, we now examine how performance changes over time.

Trend analysis helps us understand whether municipal demand and service performance are improving, worsening, or remaining stable.

We will analyze:

- monthly request volume;
- monthly requests by directorate;
- monthly requests by area.

This allows us to identify where the strongest changes occur.

In [ ]:
# 1. Total requests by month
monthly_requests = (
    kerkesat_clean
    .groupby("Muaji")
    .size()
    .reset_index(name="total_requests")
)

display(monthly_requests)


# 2. Requests by month and directorate
monthly_by_directorate = (
    kerkesat_clean
    .dropna(subset=["Drejtoria"])
    .groupby(
        ["Muaji", "Drejtoria"]
    )
    .size()
    .reset_index(name="requests")
)

display(monthly_by_directorate)


# 3. Requests by month and area
monthly_by_area = (
    kerkesat_clean
    .groupby(
        ["Muaji", "Lagjja / Fshati"]
    )
    .size()
    .reset_index(name="requests")
)

display(monthly_by_area)

,Muaji,total_requests
0,2026-06,107
1,2026-07,121
2,2026-08,136
3,2026-09,192


,Muaji,Drejtoria,requests
0,2026-06,DI,43
1,2026-06,DSHP,62
2,2026-07,DI,47
3,2026-07,DSHP,69
4,2026-08,DI,50
5,2026-08,DSHP,77
6,2026-09,DI,83
7,2026-09,DSHP,97


,Muaji,Lagjja / Fshati,requests
0,2026-06,Bradashesh,20
1,2026-06,Lagjja nr. 2,22
2,2026-06,Lagjja nr. 3,25
3,2026-06,Lagjja nr. 5,22
4,2026-06,Papër,18
5,2026-07,Bradashesh,25
6,2026-07,Lagjja nr. 2,24
7,2026-07,Lagjja nr. 3,30
8,2026-07,Lagjja nr. 5,22
9,2026-07,Papër,20


### Interpretation

The monthly trend shows a clear increase in citizen requests between June and September 2026.

Total requests increased from 107 in June to 192 in September.

Both directorates experienced higher demand:

- DI increased from 43 requests in June to 83 in September.
- DSHP increased from 62 requests in June to 97 in September.

The strongest area-level increase occurred in `Lagjja nr. 3`, where requests rose from 25 in June to 64 in September.

`Lagjja nr. 5` also showed a significant increase, from 22 requests in June to 45 in September.

Bradashesh increased more moderately, while Papërr remained relatively stable.

These results suggest that the overall September increase is not evenly distributed across all areas. The strongest growth is concentrated in Lagjja nr. 3 and Lagjja nr. 5, which should be examined more closely in later cross-source analysis.

## 23. Analyze SLA performance

After validating and recalculating the SLA fields, we can now measure how well requests are being resolved within their expected deadlines.

We will analyze SLA performance:

- overall;
- by month;
- by directorate;
- by area.

This helps us identify where delays are concentrated instead of looking only at the overall SLA compliance rate.

In [ ]:
# Only resolved requests can be evaluated for SLA compliance
resolved_requests_df = kerkesat_clean[
    kerkesat_clean["Statusi"] == "E zgjidhur"
].copy()


# 1. Overall SLA performance
overall_sla = (
    resolved_requests_df["Brenda_afatit_clean"]
    .value_counts()
    .rename_axis("within_sla")
    .reset_index(name="requests")
)

overall_sla["percentage"] = (
    overall_sla["requests"]
    / overall_sla["requests"].sum()
    * 100
).round(2)

display(overall_sla)


# 2. SLA by month
sla_by_month = (
    resolved_requests_df
    .groupby("Muaji")
    .agg(
        resolved_requests=("ID", "count"),
        within_sla=(
            "Brenda_afatit_clean",
            lambda x: (x == True).sum()
        ),
    )
    .reset_index()
)

sla_by_month["sla_compliance_pct"] = (
    sla_by_month["within_sla"]
    / sla_by_month["resolved_requests"]
    * 100
).round(2)

display(sla_by_month)


# 3. SLA by directorate
sla_by_directorate = (
    resolved_requests_df
    .dropna(subset=["Drejtoria"])
    .groupby("Drejtoria")
    .agg(
        resolved_requests=("ID", "count"),
        within_sla=(
            "Brenda_afatit_clean",
            lambda x: (x == True).sum()
        ),
    )
    .reset_index()
)

sla_by_directorate["sla_compliance_pct"] = (
    sla_by_directorate["within_sla"]
    / sla_by_directorate["resolved_requests"]
    * 100
).round(2)

display(sla_by_directorate)


# 4. SLA by area
sla_by_area = (
    resolved_requests_df
    .groupby("Lagjja / Fshati")
    .agg(
        resolved_requests=("ID", "count"),
        within_sla=(
            "Brenda_afatit_clean",
            lambda x: (x == True).sum()
        ),
    )
    .reset_index()
)

sla_by_area["sla_compliance_pct"] = (
    sla_by_area["within_sla"]
    / sla_by_area["resolved_requests"]
    * 100
).round(2)

sla_by_area = sla_by_area.sort_values(
    "sla_compliance_pct"
)

display(sla_by_area)

,within_sla,requests,percentage
0,True,421,87.34
1,False,61,12.66


,Muaji,resolved_requests,within_sla,sla_compliance_pct
0,2026-06,105,100,95.24
1,2026-07,116,112,96.55
2,2026-08,127,114,89.76
3,2026-09,134,95,70.90


,Drejtoria,resolved_requests,within_sla,sla_compliance_pct
0,DI,204,182,89.22
1,DSHP,278,239,85.97


,Lagjja / Fshati,resolved_requests,within_sla,sla_compliance_pct
2,Lagjja nr. 3,136,103,75.74
3,Lagjja nr. 5,99,86,86.87
0,Bradashesh,86,75,87.21
4,Papër,76,74,97.37
1,Lagjja nr. 2,85,83,97.65


### Interpretation

Overall SLA compliance is 87.34%, meaning that most resolved requests were completed within their expected deadlines.

However, SLA performance deteriorated significantly over time:

- June: 95.24%
- July: 96.55%
- August: 89.76%
- September: 70.90%

This shows that September experienced a major decline in service timeliness.

At directorate level:

- DI achieved 89.22% SLA compliance
- DSHP achieved 85.97%

At area level, the weakest performance was observed in `Lagjja nr. 3`, with only 75.74% of resolved requests completed within SLA.

`Lagjja nr. 5` and `Bradashesh` also performed below the best-performing areas, while `Papërr` and `Lagjja nr. 2` remained above 97%.

This indicates that the overall SLA decline is concentrated mainly in September and particularly in Lagjja nr. 3.

## 24. Analyze recurring problems

Recurring requests are an important signal because a request can be marked as resolved while the underlying issue later appears again.

This analysis supports the `Closed ≠ Solved` concept.

We will examine recurring problems by:

- recurring group;
- area;
- category;
- number of repeated reports;
- number of resolved cases;
- first and last report dates.

This helps identify locations where the same municipal problem continues to return over time.

In [ ]:
recurring_analysis = (
    kerkesat_clean[
        kerkesat_clean["Is_recurring"]
    ]
    .groupby(
        [
            "Recurring group",
            "Lagjja / Fshati",
            "Vendi",
            "Kategoria",
        ],
        dropna=False,
    )
    .agg(
        total_reports=("ID", "count"),
        resolved_reports=(
            "Statusi",
            lambda x: (x == "E zgjidhur").sum()
        ),
        first_report=("Data e regjistrimit", "min"),
        last_report=("Data e regjistrimit", "max"),
    )
    .reset_index()
)

recurring_analysis["resolved_pct"] = (
    recurring_analysis["resolved_reports"]
    / recurring_analysis["total_reports"]
    * 100
).round(2)

recurring_analysis = recurring_analysis.sort_values(
    "total_reports",
    ascending=False
)

display(recurring_analysis)

,Recurring group,Lagjja / Fshati,Vendi,Kategoria,total_reports,resolved_reports,first_report,last_report,resolved_pct
0,REC-001,Lagjja nr. 3,Tregu i lagjes,Mbetje të pagrumbulluara,6,6,2026-06-24,2026-09-27,100.00
1,REC-002,Lagjja nr. 3,Pallati nr. 12,Mbetje të pagrumbulluara,6,5,2026-06-25,2026-09-29,83.33
2,REC-003,Lagjja nr. 5,Kryqëzimi kryesor,Kullim / ujëra atmosferike,6,5,2026-06-08,2026-09-26,83.33
3,REC-004,Bradashesh,Rruga kryesore,Gropa / rrugë e dëmtuar,6,5,2026-08-11,2026-09-30,83.33


In [ ]:
recurring_kpis = pd.DataFrame({
    "KPI": [
        "Recurring requests",
        "Recurring issue groups",
        "Areas affected",
    ],
    "Value": [
        int(kerkesat_clean["Is_recurring"].sum()),
        kerkesat_clean["Recurring group"].nunique(),
        kerkesat_clean.loc[
            kerkesat_clean["Is_recurring"],
            "Lagjja / Fshati"
        ].nunique(),
    ],
})

display(recurring_kpis)

,KPI,Value
0,Recurring requests,24
1,Recurring issue groups,4
2,Areas affected,3


### Interpretation

The recurring-problem analysis identified 24 recurring requests grouped into 4 recurring issue groups across 3 affected areas.

The strongest recurring patterns are:

- `REC-001` — Lagjja nr. 3, Tregu i lagjes, waste collection issue:
  6 reports, all 6 marked as resolved.

- `REC-002` — Lagjja nr. 3, Pallati nr. 12, waste collection issue:
  6 reports, 5 marked as resolved.

- `REC-003` — Lagjja nr. 5, Kryqëzimi kryesor, drainage issue:
  6 reports, 5 marked as resolved.

- `REC-004` — Bradashesh, Rruga kryesore, road damage:
  6 reports, 5 marked as resolved.

This is an important decision-support signal because several problems were repeatedly reported even after previous cases had been marked as resolved.

The `Closed ≠ Solved` pattern is therefore clearly present in the dataset and should be considered when prioritizing areas for deeper review.

## 25. Cross-source analysis

The main value of Luma comes from combining information from multiple municipal data sources.

A rise in citizen requests is more meaningful when it is supported by operational evidence.

For example:

- more waste complaints become more important if missed collections and delays are also increasing;
- more road complaints become more important if open incidents and repair time are also increasing;
- service pressure becomes more important when it is compared with budget execution.

In this step, we combine citizen-demand indicators with operational and financial indicators to create a more complete view of municipal performance.

In [ ]:
# -----------------------------
# DSHP / Waste cross-source view
# -----------------------------

# Citizen requests for DSHP by month and area
dshp_requests = (
    kerkesat_clean[
        kerkesat_clean["Drejtoria"] == "DSHP"
    ]
    .groupby(
        ["Muaji", "Lagjja / Fshati"]
    )
    .size()
    .reset_index(name="citizen_requests")
)

# Aggregate waste operations by month and area
waste_monthly = (
    mbetje_clean
    .groupby(
        ["month", "area"]
    )
    .agg(
        scheduled_collections=(
            "scheduled_collections",
            "sum"
        ),
        completed_collections=(
            "completed_collections",
            "sum"
        ),
        missed_collections=(
            "missed_collections",
            "sum"
        ),
        avg_delay_hours=(
            "avg_delay_hours",
            "mean"
        ),
        vehicle_availability_pct=(
            "vehicle_availability_pct",
            "mean"
        ),
    )
    .reset_index()
)

# Make month column names compatible
dshp_requests = dshp_requests.rename(
    columns={
        "Muaji": "month",
        "Lagjja / Fshati": "area"
    }
)

# Merge requests + waste operations
dshp_cross_source = pd.merge(
    dshp_requests,
    waste_monthly,
    on=["month", "area"],
    how="left"
)

display(
    dshp_cross_source.sort_values(
        ["area", "month"]
    )
)

,month,area,citizen_requests,scheduled_collections,completed_collections,missed_collections,avg_delay_hours,vehicle_availability_pct
0,2026-06,Bradashesh,11,433,419,14,1.250,90.175
5,2026-07,Bradashesh,15,446,430,16,1.275,93.150
10,2026-08,Bradashesh,11,444,430,14,1.375,91.850
15,2026-09,Bradashesh,11,444,407,37,3.475,84.850
1,2026-06,Lagjja nr. 2,12,477,459,18,1.075,93.075
6,2026-07,Lagjja nr. 2,14,465,452,13,1.575,94.925
11,2026-08,Lagjja nr. 2,14,490,475,15,1.025,91.475
16,2026-09,Lagjja nr. 2,13,437,423,14,1.125,92.250
2,2026-06,Lagjja nr. 3,16,463,447,16,1.325,90.525
7,2026-07,Lagjja nr. 3,18,504,488,16,1.300,92.650


In [ ]:
# -----------------------------
# DI / Roads cross-source view
# -----------------------------

di_requests = (
    kerkesat_clean[
        kerkesat_clean["Drejtoria"] == "DI"
    ]
    .groupby(
        ["Muaji", "Lagjja / Fshati"]
    )
    .size()
    .reset_index(name="citizen_requests")
)

di_requests = di_requests.rename(
    columns={
        "Muaji": "month",
        "Lagjja / Fshati": "area"
    }
)

roads_monthly = (
    rruge_clean
    .groupby(
        ["month", "area"]
    )
    .agg(
        reported_incidents=(
            "reported_incidents",
            "sum"
        ),
        open_incidents=(
            "open_incidents",
            "sum"
        ),
        avg_repair_days=(
            "avg_repair_days",
            "mean"
        ),
        drainage_cases=(
            "drainage_cases",
            "sum"
        ),
        emergency_repairs=(
            "emergency_repairs",
            "sum"
        ),
    )
    .reset_index()
)

di_cross_source = pd.merge(
    di_requests,
    roads_monthly,
    on=["month", "area"],
    how="left"
)

display(
    di_cross_source.sort_values(
        ["area", "month"]
    )
)

,month,area,citizen_requests,reported_incidents,open_incidents,avg_repair_days,drainage_cases,emergency_repairs
0,2026-06,Bradashesh,8,11,1,5.2,3,3
5,2026-07,Bradashesh,8,13,4,3.9,1,1
10,2026-08,Bradashesh,11,13,3,5.4,4,3
15,2026-09,Bradashesh,20,21,8,7.8,9,5
1,2026-06,Lagjja nr. 2,9,12,1,4.0,1,3
6,2026-07,Lagjja nr. 2,8,13,4,4.6,4,2
11,2026-08,Lagjja nr. 2,8,7,4,4.0,1,2
16,2026-09,Lagjja nr. 2,12,14,3,3.2,4,0
2,2026-06,Lagjja nr. 3,9,14,2,4.1,2,1
7,2026-07,Lagjja nr. 3,11,14,4,5.7,1,1


In [ ]:
# Budget context by month and directorate

budget_context = buxhet_clean[
    [
        "month",
        "directorate",
        "execution_pct"
    ]
].copy()

display(budget_context)

,month,directorate,execution_pct
0,2026-06,DI,71.80
1,2026-06,DSHP,80.24
2,2026-07,DI,72.75
3,2026-07,DSHP,81.19
4,2026-08,DI,75.14
5,2026-08,DSHP,92.44
6,2026-09,DI,76.96
7,2026-09,DSHP,102.29


### Interpretation

Cross-source analysis shows that service deterioration is concentrated in specific areas rather than occurring uniformly across the municipality.

For DI, Lagjja nr. 5 shows the strongest deterioration in September. Citizen requests increased substantially, while reported incidents, open incidents, drainage cases and average repair time also reached their highest levels.

Bradashesh also shows worsening infrastructure performance in September, but at a lower level than Lagjja nr. 5.

Papërr remains relatively stable and can therefore serve as a useful baseline for comparison.

The budget data adds an additional layer of context. DSHP budget execution increased to 102.29% in September, while DI remained below 80%.

These combined signals provide stronger evidence than any single dataset alone and form the basis for priority detection.

## 26. Detect priority areas

The goal of priority detection is to identify where municipal performance is deteriorating enough to require management attention.

Instead of using only one metric, we combine multiple signals.

For DSHP we consider:

- growth in citizen requests;
- increase in missed collections;
- increase in operational delays;
- decrease in vehicle availability;
- budget pressure.

For DI we consider:

- growth in citizen requests;
- increase in reported incidents;
- increase in open incidents;
- increase in average repair time;
- increase in drainage cases.

Each triggered signal contributes to the final priority level:

- **0–1 signals → Low**
- **2–3 signals → Medium**
- **4 or more signals → High**

This produces a simple and explainable prioritization model.

In [ ]:
def priority_from_signals(signal_count):
    if signal_count >= 4:
        return "High"
    elif signal_count >= 2:
        return "Medium"
    else:
        return "Low"

In [ ]:
# Compare August vs September for DSHP

dshp_aug = dshp_cross_source[
    dshp_cross_source["month"] == pd.Period("2026-08", freq="M")
].copy()

dshp_sep = dshp_cross_source[
    dshp_cross_source["month"] == pd.Period("2026-09", freq="M")
].copy()

dshp_priority = pd.merge(
    dshp_aug,
    dshp_sep,
    on="area",
    suffixes=("_aug", "_sep")
)

dshp_priority["request_growth_pct"] = (
    (
        dshp_priority["citizen_requests_sep"]
        - dshp_priority["citizen_requests_aug"]
    )
    / dshp_priority["citizen_requests_aug"]
    * 100
).round(2)

dshp_priority["missed_growth_pct"] = (
    (
        dshp_priority["missed_collections_sep"]
        - dshp_priority["missed_collections_aug"]
    )
    / dshp_priority["missed_collections_aug"]
    * 100
).round(2)

dshp_priority["delay_growth_pct"] = (
    (
        dshp_priority["avg_delay_hours_sep"]
        - dshp_priority["avg_delay_hours_aug"]
    )
    / dshp_priority["avg_delay_hours_aug"]
    * 100
).round(2)

dshp_priority["vehicle_change_pp"] = (
    dshp_priority["vehicle_availability_pct_sep"]
    - dshp_priority["vehicle_availability_pct_aug"]
).round(2)

# Trigger rules
dshp_priority["signal_requests"] = (
    dshp_priority["request_growth_pct"] > 20
)

dshp_priority["signal_missed"] = (
    dshp_priority["missed_growth_pct"] > 20
)

dshp_priority["signal_delay"] = (
    dshp_priority["delay_growth_pct"] > 20
)

dshp_priority["signal_vehicle"] = (
    dshp_priority["vehicle_change_pp"] < -5
)

# Budget signal is directorate-level
dshp_sep_budget = buxhet_clean[
    (buxhet_clean["month"] == pd.Period("2026-09", freq="M"))
    & (buxhet_clean["directorate"] == "DSHP")
]["execution_pct"].iloc[0]

dshp_priority["signal_budget"] = (
    dshp_sep_budget >= 100
)

signal_cols_dshp = [
    "signal_requests",
    "signal_missed",
    "signal_delay",
    "signal_vehicle",
    "signal_budget",
]

dshp_priority["signal_count"] = (
    dshp_priority[signal_cols_dshp]
    .sum(axis=1)
)

dshp_priority["priority"] = (
    dshp_priority["signal_count"]
    .apply(priority_from_signals)
)

display(
    dshp_priority[
        [
            "area",
            "request_growth_pct",
            "missed_growth_pct",
            "delay_growth_pct",
            "vehicle_change_pp",
            "signal_count",
            "priority",
        ]
    ].sort_values("signal_count", ascending=False)
)

,area,request_growth_pct,missed_growth_pct,delay_growth_pct,vehicle_change_pp,signal_count,priority
2,Lagjja nr. 3,50.00,47.89,45.91,-8.33,5,High
0,Bradashesh,0.00,164.29,152.73,-7.00,4,High
3,Lagjja nr. 5,30.00,-17.65,13.46,0.25,2,Medium
1,Lagjja nr. 2,-7.14,-6.67,9.76,0.78,1,Low
4,Papër,20.00,-12.50,9.09,-4.48,1,Low


In [ ]:
# Compare August vs September for DI

di_aug = di_cross_source[
    di_cross_source["month"] == pd.Period("2026-08", freq="M")
].copy()

di_sep = di_cross_source[
    di_cross_source["month"] == pd.Period("2026-09", freq="M")
].copy()

di_priority = pd.merge(
    di_aug,
    di_sep,
    on="area",
    suffixes=("_aug", "_sep")
)

di_priority["request_growth_pct"] = (
    (
        di_priority["citizen_requests_sep"]
        - di_priority["citizen_requests_aug"]
    )
    / di_priority["citizen_requests_aug"]
    * 100
).round(2)

di_priority["reported_incidents_growth_pct"] = (
    (
        di_priority["reported_incidents_sep"]
        - di_priority["reported_incidents_aug"]
    )
    / di_priority["reported_incidents_aug"]
    * 100
).round(2)

di_priority["open_incidents_growth_pct"] = (
    (
        di_priority["open_incidents_sep"]
        - di_priority["open_incidents_aug"]
    )
    / di_priority["open_incidents_aug"]
    * 100
).round(2)

di_priority["repair_time_growth_pct"] = (
    (
        di_priority["avg_repair_days_sep"]
        - di_priority["avg_repair_days_aug"]
    )
    / di_priority["avg_repair_days_aug"]
    * 100
).round(2)

di_priority["drainage_growth_pct"] = (
    (
        di_priority["drainage_cases_sep"]
        - di_priority["drainage_cases_aug"]
    )
    / di_priority["drainage_cases_aug"]
    * 100
).round(2)

# Trigger rules
di_priority["signal_requests"] = (
    di_priority["request_growth_pct"] > 20
)

di_priority["signal_reported"] = (
    di_priority["reported_incidents_growth_pct"] > 20
)

di_priority["signal_open"] = (
    di_priority["open_incidents_growth_pct"] > 20
)

di_priority["signal_repair"] = (
    di_priority["repair_time_growth_pct"] > 20
)

di_priority["signal_drainage"] = (
    di_priority["drainage_growth_pct"] > 20
)

signal_cols_di = [
    "signal_requests",
    "signal_reported",
    "signal_open",
    "signal_repair",
    "signal_drainage",
]

di_priority["signal_count"] = (
    di_priority[signal_cols_di]
    .sum(axis=1)
)

di_priority["priority"] = (
    di_priority["signal_count"]
    .apply(priority_from_signals)
)

display(
    di_priority[
        [
            "area",
            "request_growth_pct",
            "reported_incidents_growth_pct",
            "open_incidents_growth_pct",
            "repair_time_growth_pct",
            "drainage_growth_pct",
            "signal_count",
            "priority",
        ]
    ].sort_values("signal_count", ascending=False)
)


,area,request_growth_pct,reported_incidents_growth_pct,open_incidents_growth_pct,repair_time_growth_pct,drainage_growth_pct,signal_count,priority
0,Bradashesh,81.82,61.54,166.67,44.44,125.00,5,High
3,Lagjja nr. 5,107.14,86.67,140.00,44.62,116.67,5,High
1,Lagjja nr. 2,50.00,100.00,-25.00,-20.00,300.00,3,Medium
2,Lagjja nr. 3,50.00,33.33,0.00,-12.77,-33.33,2,Medium
4,Papër,11.11,-22.22,0.00,4.65,100.00,1,Low


### Interpretation

The priority-detection model identified different levels of service pressure across areas and directorates.

For DSHP:

- `Lagjja nr. 3` is classified as **High priority**, with all five signals triggered.
- `Bradashesh` is also classified as **High priority**, driven mainly by strong increases in missed collections, delays and lower vehicle availability.
- `Lagjja nr. 5` is classified as **Medium priority**.
- `Lagjja nr. 2` and `Papër` remain **Low priority**.

For DI:

- `Lagjja nr. 5` and `Bradashesh` are both classified as **High priority**, with all five deterioration signals triggered.
- `Lagjja nr. 2` and `Lagjja nr. 3` are classified as **Medium priority**.
- `Papër` remains **Low priority**.

The results show that municipal performance problems are not concentrated in only one location.

Different areas can become high priority for different operational reasons, which is exactly why Luma combines multiple indicators instead of relying on a single KPI.

## 27. Build structured decision insights

The priority tables identify which areas require attention, but the dashboard and AI agent need a more structured output.

For each high- or medium-priority case, we create a compact decision-support record containing:

- area;
- directorate;
- priority level;
- number of triggered signals;
- key evidence;
- a short finding;
- a recommended next step.

The goal is not to automate the decision itself.

The system organizes the evidence and suggests what management should review first, while the final decision remains with municipal staff.

In [ ]:
def build_dshp_insights(df):
    insights = []

    for _, row in df.iterrows():

        if row["priority"] == "Low":
            continue

        evidence = {
            "request_growth_pct": row["request_growth_pct"],
            "missed_collections_growth_pct": row["missed_growth_pct"],
            "delay_growth_pct": row["delay_growth_pct"],
            "vehicle_availability_change_pp": row["vehicle_change_pp"],
        }

        insight = {
            "area": row["area"],
            "directorate": "DSHP",
            "priority": row["priority"],
            "signal_count": int(row["signal_count"]),
            "evidence": evidence,
            "finding": (
                f"{row['area']} shows deterioration in public-service "
                f"performance based on multiple citizen and operational signals."
            ),
            "recommended_next_step": (
                "Review missed collections, service delays and vehicle "
                "availability before deciding on operational action."
            ),
        }

        insights.append(insight)

    return insights


def build_di_insights(df):
    insights = []

    for _, row in df.iterrows():

        if row["priority"] == "Low":
            continue

        evidence = {
            "request_growth_pct": row["request_growth_pct"],
            "reported_incidents_growth_pct":
                row["reported_incidents_growth_pct"],
            "open_incidents_growth_pct":
                row["open_incidents_growth_pct"],
            "repair_time_growth_pct":
                row["repair_time_growth_pct"],
            "drainage_growth_pct":
                row["drainage_growth_pct"],
        }

        insight = {
            "area": row["area"],
            "directorate": "DI",
            "priority": row["priority"],
            "signal_count": int(row["signal_count"]),
            "evidence": evidence,
            "finding": (
                f"{row['area']} shows deterioration in infrastructure "
                f"performance based on multiple citizen and operational signals."
            ),
            "recommended_next_step": (
                "Review open incidents, repair delays and drainage cases "
                "to identify the operational bottleneck."
            ),
        }

        insights.append(insight)

    return insights


dshp_insights = build_dshp_insights(dshp_priority)
di_insights = build_di_insights(di_priority)

structured_insights = dshp_insights + di_insights

In [ ]:
for insight in structured_insights:
    print("=" * 70)
    print("AREA:", insight["area"])
    print("DIRECTORATE:", insight["directorate"])
    print("PRIORITY:", insight["priority"])
    print("SIGNALS:", insight["signal_count"])
    print("FINDING:", insight["finding"])
    print("RECOMMENDED NEXT STEP:", insight["recommended_next_step"])
    print("EVIDENCE:", insight["evidence"])
    print()

AREA: Bradashesh
DIRECTORATE: DSHP
PRIORITY: High
SIGNALS: 4
FINDING: Bradashesh shows deterioration in public-service performance based on multiple citizen and operational signals.
RECOMMENDED NEXT STEP: Review missed collections, service delays and vehicle availability before deciding on operational action.
EVIDENCE: {'request_growth_pct': 0.0, 'missed_collections_growth_pct': 164.29, 'delay_growth_pct': 152.73, 'vehicle_availability_change_pp': -7.0}

AREA: Lagjja nr. 3
DIRECTORATE: DSHP
PRIORITY: High
SIGNALS: 5
FINDING: Lagjja nr. 3 shows deterioration in public-service performance based on multiple citizen and operational signals.
RECOMMENDED NEXT STEP: Review missed collections, service delays and vehicle availability before deciding on operational action.
EVIDENCE: {'request_growth_pct': 50.0, 'missed_collections_growth_pct': 47.89, 'delay_growth_pct': 45.91, 'vehicle_availability_change_pp': -8.33}

AREA: Lagjja nr. 5
DIRECTORATE: DSHP
PRIORITY: Medium
SIGNALS: 2
FINDING: Lagj

## 28. Prepare outputs for backend and dashboard

The final step is to prepare compact, structured outputs that can be consumed by the FastAPI backend, dashboard components and Luma Agent.

The notebook should not send entire raw DataFrames to the frontend.

Instead, we prepare specific outputs for:

- overall KPIs;
- monthly trends;
- SLA performance;
- recurring issues;
- budget execution;
- priority areas;
- structured decision insights.

These outputs can later be reproduced inside backend services and exposed through API endpoints.

In [ ]:
# 1. Overall KPI output
backend_kpis = {
    "total_requests": int(total_requests),
    "active_requests": int(active_requests),
    "resolved_requests": int(resolved_requests),
    "resolution_rate_pct": round(float(resolution_rate), 2),
    "avg_resolution_days": round(float(avg_resolution_days), 2),
    "sla_compliance_pct": round(float(sla_compliance_rate), 2),
    "recurring_requests": int(recurring_requests),
}

backend_kpis

{'total_requests': 556,
 'active_requests': 74,
 'resolved_requests': 482,
 'resolution_rate_pct': 86.69,
 'avg_resolution_days': 4.9,
 'sla_compliance_pct': 87.34,
 'recurring_requests': 24}

In [ ]:
# 2. Monthly trend output
backend_monthly_trend = (
    monthly_requests
    .assign(
        Muaji=lambda df: df["Muaji"].astype(str)
    )
    .rename(
        columns={
            "Muaji": "month",
            "total_requests": "requests"
        }
    )
    .to_dict(orient="records")
)

backend_monthly_trend

[{'month': '2026-06', 'requests': 107},
 {'month': '2026-07', 'requests': 121},
 {'month': '2026-08', 'requests': 136},
 {'month': '2026-09', 'requests': 192}]

In [ ]:
# 3. SLA output by area
backend_sla_by_area = (
    sla_by_area
    .rename(
        columns={
            "Lagjja / Fshati": "area"
        }
    )
    .to_dict(orient="records")
)

backend_sla_by_area

[{'area': 'Lagjja nr. 3',
  'resolved_requests': 136,
  'within_sla': 103,
  'sla_compliance_pct': 75.74},
 {'area': 'Lagjja nr. 5',
  'resolved_requests': 99,
  'within_sla': 86,
  'sla_compliance_pct': 86.87},
 {'area': 'Bradashesh',
  'resolved_requests': 86,
  'within_sla': 75,
  'sla_compliance_pct': 87.21},
 {'area': 'Papër',
  'resolved_requests': 76,
  'within_sla': 74,
  'sla_compliance_pct': 97.37},
 {'area': 'Lagjja nr. 2',
  'resolved_requests': 85,
  'within_sla': 83,
  'sla_compliance_pct': 97.65}]

In [ ]:
# 4. Recurring issues output
backend_recurring_issues = (
    recurring_analysis
    .assign(
        first_report=lambda df:
            df["first_report"].dt.strftime("%Y-%m-%d"),

        last_report=lambda df:
            df["last_report"].dt.strftime("%Y-%m-%d"),
    )
    .rename(
        columns={
            "Recurring group": "recurring_group",
            "Lagjja / Fshati": "area",
            "Vendi": "location",
            "Kategoria": "category",
        }
    )
    .to_dict(orient="records")
)

backend_recurring_issues

[{'recurring_group': 'REC-001',
  'area': 'Lagjja nr. 3',
  'location': 'Tregu i lagjes',
  'category': 'Mbetje të pagrumbulluara',
  'total_reports': 6,
  'resolved_reports': 6,
  'first_report': '2026-06-24',
  'last_report': '2026-09-27',
  'resolved_pct': 100.0},
 {'recurring_group': 'REC-002',
  'area': 'Lagjja nr. 3',
  'location': 'Pallati nr. 12',
  'category': 'Mbetje të pagrumbulluara',
  'total_reports': 6,
  'resolved_reports': 5,
  'first_report': '2026-06-25',
  'last_report': '2026-09-29',
  'resolved_pct': 83.33},
 {'recurring_group': 'REC-003',
  'area': 'Lagjja nr. 5',
  'location': 'Kryqëzimi kryesor',
  'category': 'Kullim / ujëra atmosferike',
  'total_reports': 6,
  'resolved_reports': 5,
  'first_report': '2026-06-08',
  'last_report': '2026-09-26',
  'resolved_pct': 83.33},
 {'recurring_group': 'REC-004',
  'area': 'Bradashesh',
  'location': 'Rruga kryesore',
  'category': 'Gropa / rrugë e dëmtuar',
  'total_reports': 6,
  'resolved_reports': 5,
  'first_report

In [ ]:
# 5. Budget output
backend_budget = (
    buxhet_clean[
        [
            "month",
            "directorate",
            "budget_allocated_all",
            "actual_spend_all",
            "execution_pct",
        ]
    ]
    .assign(
        month=lambda df:
            df["month"].astype(str)
    )
    .to_dict(orient="records")
)

backend_budget

[{'month': '2026-06',
  'directorate': 'DI',
  'budget_allocated_all': 50000000,
  'actual_spend_all': 35900000,
  'execution_pct': 71.8},
 {'month': '2026-06',
  'directorate': 'DSHP',
  'budget_allocated_all': 41000000,
  'actual_spend_all': 32900000,
  'execution_pct': 80.24},
 {'month': '2026-07',
  'directorate': 'DI',
  'budget_allocated_all': 51000000,
  'actual_spend_all': 37100000,
  'execution_pct': 72.75},
 {'month': '2026-07',
  'directorate': 'DSHP',
  'budget_allocated_all': 42000000,
  'actual_spend_all': 34100000,
  'execution_pct': 81.19},
 {'month': '2026-08',
  'directorate': 'DI',
  'budget_allocated_all': 53500000,
  'actual_spend_all': 40200000,
  'execution_pct': 75.14},
 {'month': '2026-08',
  'directorate': 'DSHP',
  'budget_allocated_all': 45000000,
  'actual_spend_all': 41600000,
  'execution_pct': 92.44},
 {'month': '2026-09',
  'directorate': 'DI',
  'budget_allocated_all': 56000000,
  'actual_spend_all': 43100000,
  'execution_pct': 76.96},
 {'month': '202

In [ ]:
# 6. Priority output

dshp_priority_output = (
    dshp_priority[
        [
            "area",
            "signal_count",
            "priority",
            "request_growth_pct",
            "missed_growth_pct",
            "delay_growth_pct",
            "vehicle_change_pp",
        ]
    ]
    .copy()
)

dshp_priority_output["directorate"] = "DSHP"


di_priority_output = (
    di_priority[
        [
            "area",
            "signal_count",
            "priority",
            "request_growth_pct",
            "reported_incidents_growth_pct",
            "open_incidents_growth_pct",
            "repair_time_growth_pct",
            "drainage_growth_pct",
        ]
    ]
    .copy()
)

di_priority_output["directorate"] = "DI"


backend_priorities = {
    "DSHP": dshp_priority_output.to_dict(
        orient="records"
    ),
    "DI": di_priority_output.to_dict(
        orient="records"
    ),
}

backend_priorities

{'DSHP': [{'area': 'Bradashesh',
   'signal_count': 4,
   'priority': 'High',
   'request_growth_pct': 0.0,
   'missed_growth_pct': 164.29,
   'delay_growth_pct': 152.73,
   'vehicle_change_pp': -7.0,
   'directorate': 'DSHP'},
  {'area': 'Lagjja nr. 2',
   'signal_count': 1,
   'priority': 'Low',
   'request_growth_pct': -7.14,
   'missed_growth_pct': -6.67,
   'delay_growth_pct': 9.76,
   'vehicle_change_pp': 0.78,
   'directorate': 'DSHP'},
  {'area': 'Lagjja nr. 3',
   'signal_count': 5,
   'priority': 'High',
   'request_growth_pct': 50.0,
   'missed_growth_pct': 47.89,
   'delay_growth_pct': 45.91,
   'vehicle_change_pp': -8.33,
   'directorate': 'DSHP'},
  {'area': 'Lagjja nr. 5',
   'signal_count': 2,
   'priority': 'Medium',
   'request_growth_pct': 30.0,
   'missed_growth_pct': -17.65,
   'delay_growth_pct': 13.46,
   'vehicle_change_pp': 0.25,
   'directorate': 'DSHP'},
  {'area': 'Papër',
   'signal_count': 1,
   'priority': 'Low',
   'request_growth_pct': 20.0,
   'missed_

In [ ]:
# 7. Structured decision insights
backend_decision_insights = structured_insights

backend_decision_insights

[{'area': 'Bradashesh',
  'directorate': 'DSHP',
  'priority': 'High',
  'signal_count': 4,
  'evidence': {'request_growth_pct': 0.0,
   'missed_collections_growth_pct': 164.29,
   'delay_growth_pct': 152.73,
   'vehicle_availability_change_pp': -7.0},
  'finding': 'Bradashesh shows deterioration in public-service performance based on multiple citizen and operational signals.',
  'recommended_next_step': 'Review missed collections, service delays and vehicle availability before deciding on operational action.'},
 {'area': 'Lagjja nr. 3',
  'directorate': 'DSHP',
  'priority': 'High',
  'signal_count': 5,
  'evidence': {'request_growth_pct': 50.0,
   'missed_collections_growth_pct': 47.89,
   'delay_growth_pct': 45.91,
   'vehicle_availability_change_pp': -8.33},
  'finding': 'Lagjja nr. 3 shows deterioration in public-service performance based on multiple citizen and operational signals.',
  'recommended_next_step': 'Review missed collections, service delays and vehicle availability be

In [ ]:
backend_output = {
    "kpis": backend_kpis,
    "monthly_trend": backend_monthly_trend,
    "sla_by_area": backend_sla_by_area,
    "recurring_issues": backend_recurring_issues,
    "budget": backend_budget,
    "priorities": backend_priorities,
    "decision_insights": backend_decision_insights,
}

print("Backend-ready output created successfully.")

Backend-ready output created successfully.
